# fastAPI_Model — LoRA Training on Google Colab


## 1. Session preflight


In [3]:
from __future__ import annotations

import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Reduces allocator fragmentation, which matters because the loss head allocates
# a large short-lived tensor on every step. Must be set before torch initialises
# CUDA -- if torch is already imported, restart the runtime for it to take effect.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
print("Running in Colab:", IN_COLAB)
print("Python:", platform.python_version())
if not IN_COLAB:
    print()
    print("WARNING: this notebook expects Colab (Drive mount, /content paths).")
    print("For a local run use train_fastapi_model_local.ipynb instead.")

Running in Colab: True
Python: 3.12.13


In [ ]:
# GPU probe before anything is downloaded.
GPU_NAME = ""
GPU_VRAM_MB = 0
GPU_COMPUTE = ""

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    try:
        probe = subprocess.run(
            [nvidia_smi, "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=30,
            check=True,
        )
        parts = [p.strip() for p in probe.stdout.strip().splitlines()[0].split(",")]
        GPU_NAME, GPU_VRAM_MB = parts[0], int(float(parts[1]))
        GPU_COMPUTE = parts[2] if len(parts) > 2 else ""
    except Exception as exc:  # noqa: BLE001
        print("nvidia-smi probe failed:", exc)

if not GPU_NAME:
    raise RuntimeError(
        "No GPU detected. Runtime > Change runtime type > Hardware accelerator > T4 GPU, "
        "then re-run this cell."
    )

try:
    COMPUTE_MAJOR = int(str(GPU_COMPUTE).split(".")[0]) if GPU_COMPUTE else 0
except ValueError:
    COMPUTE_MAJOR = 0
SUPPORTS_BF16 = COMPUTE_MAJOR >= 8  # bf16 arrives with Ampere

print("GPU:", GPU_NAME)
print("VRAM (MiB):", GPU_VRAM_MB)
print("Compute capability:", GPU_COMPUTE)
print("bfloat16 usable:", SUPPORTS_BF16, "(fp16 otherwise)")
if not SUPPORTS_BF16:
    print()
    print(f"{GPU_NAME} is pre-Ampere, so training uses fp16.")
    print("This is expected on a T4 and is not a misconfiguration.")

## 2. Dependencies


In [6]:
def pip_install(packages: list[str]) -> None:
    command = [sys.executable, "-m", "pip", "install", "-q", "-U", *packages]
    print("$ pip install -q -U", " ".join(packages))
    subprocess.run(command, check=True)


# bitsandbytes and peft are the two that matter; Colab's versions lag.
pip_install(
    [
        "transformers>=4.44.0",
        "peft>=0.12.0",
        "bitsandbytes>=0.43.0",
        "accelerate>=0.34.0",
        "datasets>=2.20.0",
    ]
)

$ pip install -q -U transformers>=4.44.0 peft>=0.12.0 bitsandbytes>=0.43.0 accelerate>=0.34.0 datasets>=2.20.0


In [ ]:
import torch  # noqa: E402

print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("torch cannot see the GPU. Restart the runtime and re-run section 1.")
print("Device:", torch.cuda.get_device_name(0))

TORCH_BF16 = bool(torch.cuda.is_bf16_supported()) and SUPPORTS_BF16
COMPUTE_DTYPE = torch.bfloat16 if TORCH_BF16 else torch.float16
print("Compute dtype:", COMPUTE_DTYPE)

import bitsandbytes  # noqa: F401,E402

print("bitsandbytes:", bitsandbytes.__version__)

## 3. Mount Drive and configure


In [8]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/bot-training")
if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        f"{DRIVE_ROOT} not found. Create the folder in Drive and upload the data files "
        "in the layout described at the top of this notebook."
    )
print("Drive root:", DRIVE_ROOT)
print("Contents:", sorted(p.name for p in DRIVE_ROOT.iterdir()))

Mounted at /content/drive
Drive root: /content/drive/MyDrive/bot-training
Contents: ['fastapi_directive_dataset.jsonl', 'fastapi_directive_eval.jsonl', 'runtime_protocol_seed.jsonl', 'training_protocol.json', 'training_runs']


In [8]:
from dataclasses import asdict, dataclass, field  # noqa: E402

MEMORY_PROFILE_OVERRIDE = ""  # "" = auto


def select_memory_profile(vram_mb: int) -> str:
    if MEMORY_PROFILE_OVERRIDE:
        return MEMORY_PROFILE_OVERRIDE
    if vram_mb >= 22_000:
        return "GPU_24GB"  # A100 / L4 high tier
    if vram_mb >= 14_000:
        return "GPU_16GB"  # T4, the usual free-tier card
    if vram_mb >= 7_000:
        return "GPU_8GB"
    return "GPU_4GB"


@dataclass
class TrainConfig:
    model_name: str = "fastAPI_Model"
    base_model_id: str = "Qwen/Qwen2.5-Coder-3B-Instruct"

    use_runtime_seed: bool = True
    use_converted_legacy: bool = True
    include_synthetic_create_file: bool = False
    eval_ratio: float = 0.06
    max_records_per_task: int | None = None

    seed: int = 42
    max_seq_length: int = 2048
    num_epochs: float = 3.0
    learning_rate: float = 1.5e-4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    max_grad_norm: float = 0.3
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8

    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_targets: tuple[str, ...] = (
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    )

    logging_steps: int = 5
    save_steps: int = 50
    eval_steps: int = 50
    save_total_limit: int = 3
    early_stopping_patience: int = 4
    resume_from_checkpoint: str | None = None
    # "epoch" checkpoints once per epoch, which is what best-epoch selection needs.
    checkpoint_strategy: str = "steps"
    # The Trainer's eval pass is pure overhead beyond what early stopping needs.
    # Capping it here is the difference between minutes and tens of minutes; the
    # full eval set is still used for the generation evaluation in section 10.
    trainer_eval_max_rows: int = 256
    # Hard wall-clock cap on training. 0 disables it and epochs run to completion.
    target_train_minutes: int = 0

    eval_sample_limit: int | None = 180
    max_new_tokens: int = 1024

    merge_adapter: bool = True
    merge_on_cpu: bool = False  # 16 GB fits a 3B fp16 merge on the GPU
    export_gguf: bool = True
    quantization: str = "Q4_K_M"
    # Keep the f16 GGUF instead of deleting it after quantizing. Worth keeping:
    # re-quantizing to another level later needs only this file, not a re-merge.
    keep_f16_gguf: bool = True
    # The two large intermediates are built on Colab's local disk, which is fast.
    # Copying them to Drive is optional because each is ~6.2 GB and the Drive
    # mount writes slowly; the quantized artifact is always copied.
    copy_merged_to_drive: bool = False
    copy_f16_gguf_to_drive: bool = False

    require_min_records_per_task: int = 0

    memory_profile: str = field(default_factory=lambda: select_memory_profile(GPU_VRAM_MB))


CFG = TrainConfig()

if CFG.memory_profile == "GPU_4GB":
    CFG.max_seq_length, CFG.per_device_train_batch_size = 768, 1
    CFG.gradient_accumulation_steps = 16
    CFG.lora_r, CFG.lora_alpha = 16, 32
    CFG.eval_sample_limit = 60
    CFG.merge_on_cpu = True
elif CFG.memory_profile == "GPU_8GB":
    CFG.max_seq_length, CFG.per_device_train_batch_size = 1536, 1
    CFG.gradient_accumulation_steps = 16
    CFG.merge_on_cpu = True
elif CFG.memory_profile == "GPU_16GB":
    CFG.max_seq_length, CFG.per_device_train_batch_size = 2048, 2
    CFG.gradient_accumulation_steps = 8
elif CFG.memory_profile == "GPU_24GB":
    CFG.max_seq_length, CFG.per_device_train_batch_size = 3072, 2
    CFG.gradient_accumulation_steps = 8
    CFG.eval_sample_limit = None
else:
    raise ValueError(f"Unknown memory profile: {CFG.memory_profile}")

# ---------------------------------------------------------------------------
# Rehearsal mode
# ---------------------------------------------------------------------------
# Run this once before the full recipe. It exercises every cell end to end in a
# few minutes, so a wrong path or a missing file costs minutes instead of hours.
# The numbers it produces are real, just small: report the record count, step
# count, and epoch alongside them and they stand on their own.
REHEARSAL = False

# Wall-clock target for the training loop only, in minutes. Section 9 stops at
# the end of the first epoch that crosses it, so the run lands near the target
# instead of depending on a throughput guess. Add roughly 35-45 minutes on top
# for the base-model download, generation evaluation, merge, GGUF conversion,
# quantization, and the ~2 GB copy to Drive.
TARGET_TRAIN_MINUTES = 240

if REHEARSAL:
    CFG.max_records_per_task = 150
    CFG.num_epochs = 1.0
    CFG.max_seq_length = min(CFG.max_seq_length, 1024)
    CFG.eval_sample_limit = 18
    CFG.save_steps = CFG.eval_steps = 10
    CFG.trainer_eval_max_rows = 128
    CFG.early_stopping_patience = 3
    CFG.merge_adapter = False
    CFG.export_gguf = False
else:
    # ---------------------------------------------------------------------
    # Full run: spend the budget on unique data, not on repeats
    # ---------------------------------------------------------------------
    # Measured from the rehearsal: 29 optimizer steps took roughly 20 minutes of
    # actual training on a T4, so about 40 s per step at an effective batch of 16.
    #
    # That fixes the budget at a few hundred steps, and the only real decision is
    # what those steps see. Two ways to spend 180 minutes:
    #
    #   350 records/stage, ~4 epochs  -> 1,052 unique rows, each seen 4 times
    #   full corpus, ~1 epoch         -> 4,247 unique rows, each seen once
    #
    # The second is the better model. Repeating a small set drives training loss
    # down by memorising it; a fourfold larger set of distinct schemas, contracts,
    # and file plans is what generalises to a request the model has not seen. The
    # rehearsal already reached contract_rate 1.00 on 452 rows, so capacity is not
    # the limit -- coverage is. So: no per-stage cap.
    CFG.max_records_per_task = None  # the whole corpus, ~4,247 rows
    CFG.num_epochs = 3.0  # a ceiling; the time budget is what actually stops it

    # Keep the micro-batch at 2. This is not a conservative guess -- batch 4 runs
    # out of memory on a T4, and the reason is the vocabulary, not the model.
    #
    # Qwen2.5-Coder has a 151,936-token vocabulary, so the logits tensor for one
    # micro-batch is batch x seq x 151,936. cross_entropy upcasts it to fp32 and
    # needs a gradient buffer of the same size:
    #
    #   batch 2 x seq 1016 -> 0.62 GB fp16 logits, ~1.23 GB fp32, ~1.23 GB grad
    #   batch 4 x seq 1016 -> 1.23 GB fp16 logits, ~2.47 GB fp32, ~2.47 GB grad
    #
    # That second row is the failure: ~5 GB of transient loss memory on top of the
    # weights and optimizer state. Gradient checkpointing does not help, because
    # this is the loss head rather than the transformer activations.
    #
    # The rehearsal ran batch 2 at seq 1024 and peaked at 4.65 GiB reserved out of
    # 14.56 GiB, so batch 2 is measured-safe. Accumulation carries the effective
    # batch back to 16.
    CFG.per_device_train_batch_size = 2
    CFG.gradient_accumulation_steps = 8

    # No record in the corpus tokenizes longer than ~1,009 tokens, so 1024 drops
    # nothing. Padding is dynamic, so this is a ceiling rather than a cost -- but
    # it does bound the worst-case logits tensor above.
    CFG.max_seq_length = min(CFG.max_seq_length, 1024)

    # With roughly one epoch in budget, per-epoch checkpointing would yield a
    # single candidate and "best epoch" would be a choice of one. Checkpoint on
    # steps instead: every 40 steps is about 25 minutes, giving 7-9 evaluated
    # candidates to select the best from.
    CFG.checkpoint_strategy = "steps"
    CFG.save_steps = 40
    CFG.eval_steps = 40
    CFG.save_total_limit = 5
    CFG.early_stopping_patience = 6  # in eval events, not epochs, so allow more

    CFG.trainer_eval_max_rows = 256
    CFG.target_train_minutes = TARGET_TRAIN_MINUTES

    # Generation evaluation costs roughly 15 s per record on a T4. 90 records is
    # about 22 minutes and gives 30 per stage, which is worth the time when the
    # numbers are going into a write-up.
    CFG.eval_sample_limit = 90

    CFG.merge_adapter = True
    CFG.export_gguf = True
    CFG.keep_f16_gguf = True  # all three artifacts survive the run

RUN_SLUG = f"{CFG.model_name}-runtime-task" + ("-rehearsal" if REHEARSAL else "")

# Inputs on Drive. Both the nested and flat layouts are accepted.
TRAINING_DATA_DIR = DRIVE_ROOT / "training_data"
if not TRAINING_DATA_DIR.exists():
    TRAINING_DATA_DIR = DRIVE_ROOT
LEGACY_TRAIN_JSONL = TRAINING_DATA_DIR / "fastapi_directive_dataset.jsonl"
LEGACY_EVAL_JSONL = TRAINING_DATA_DIR / "fastapi_directive_eval.jsonl"
SEED_JSONL = TRAINING_DATA_DIR / "runtime_protocol_seed.jsonl"
PROTOCOL_JSON = DRIVE_ROOT / "training_protocol.json"

# Outputs on Drive so a disconnect does not lose the run.
RUN_ROOT = DRIVE_ROOT / "training_runs" / RUN_SLUG
BUILT_DATA_DIR = RUN_ROOT / "data"
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
ADAPTER_DIR = RUN_ROOT / "lora_adapter"
REPORT_DIR = RUN_ROOT / "reports"
EXPORT_DIR = RUN_ROOT / "export"
# Large intermediates stay on the local disk: Colab gives ~100 GB there, and
# writing a 6 GB merge through the Drive FUSE mount is painfully slow.
MERGED_DIR = Path("/content/merged_hf")
LLAMA_CPP_DIR = Path("/content/llama.cpp")

for path in (BUILT_DATA_DIR, CHECKPOINT_DIR, ADAPTER_DIR, REPORT_DIR, EXPORT_DIR, MERGED_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Memory profile:", CFG.memory_profile, "| REHEARSAL:", REHEARSAL)
print("Run root (Drive):", RUN_ROOT)
if REHEARSAL:
    print()
    print("REHEARSAL: ~28 steps, 1 epoch, 150 records/stage, seq", CFG.max_seq_length)
    print("Merge and GGUF export are skipped. Set REHEARSAL = False for the full run.")
else:
    print()
    print("=" * 78)
    print(f"FULL RUN  |  training budget {CFG.target_train_minutes} min")
    print("  corpus: no per-stage cap -- every unique record")
    print(f"  up to {CFG.num_epochs:g} epochs, seq {CFG.max_seq_length}, "
          f"batch {CFG.per_device_train_batch_size} x accum {CFG.gradient_accumulation_steps} "
          f"= {CFG.per_device_train_batch_size * CFG.gradient_accumulation_steps} effective")
    print(f"  checkpoint + eval every {CFG.save_steps} steps; "
          "best checkpoint by eval_loss is what gets exported")
    print(f"  Trainer eval capped at {CFG.trainer_eval_max_rows} rows "
          f"(generation eval uses {CFG.eval_sample_limit})")
    print("  merge -> GGUF f16 -> Q4_K_M -> Drive export bundle: ENABLED")
    print()
    print("  At the rehearsal's ~40 s/step this is roughly one pass over the whole")
    print("  corpus. One pass over every record beats several passes over a slice:")
    print("  repeats lower training loss by memorising, coverage is what generalises.")
    print()
    print("  Expect roughly:")
    print(f"    {CFG.target_train_minutes} min training")
    print("    ~5 min base-model download")
    print("    ~22 min generation evaluation (90 records)")
    print("    ~8 min merge to fp16")
    print("    ~8 min GGUF f16 conversion")
    print("    ~5 min llama-quantize build and Q4_K_M quantization")
    print("    ~6 min copying the 1.9 GB artifact to Drive")
    print(f"    total session about {CFG.target_train_minutes + 54} min "
          f"({(CFG.target_train_minutes + 54) / 60:.1f} h)")
    print()
    print("  Produces all three weight artifacts:")
    print("    merged fp16 Hugging Face weights   ~6.2 GB  (local disk)")
    print("    GGUF f16                           ~6.2 GB  (local disk)")
    print(f"    GGUF {CFG.quantization}                      ~1.9 GB  (copied to Drive)")
    print("  Section 14.1 prints the full inventory with paths.")
    if CFG.target_train_minutes > 200:
        print()
        print(f"  WARNING: a {(CFG.target_train_minutes + 54) / 60:.1f} h session is longer than")
        print("  free-tier Colab reliably allows. Checkpoints are on Drive every")
        print(f"  {CFG.save_steps} steps, so a disconnect costs time and not work -- but if you")
        print("  want the whole thing in one sitting, set TARGET_TRAIN_MINUTES = 180.")
    print()
    print("  If Colab reclaims the runtime, checkpoints are on Drive: re-run")
    print("  sections 1-8, set CFG.resume_from_checkpoint, re-run section 9.")
    print()
    print("  If section 9 raises CUDA out of memory: restart the runtime first")
    print("  (a failed allocation leaves the GPU fragmented), then set")
    print("  CFG.per_device_train_batch_size = 1 and gradient_accumulation_steps = 16.")
    print("  Effective batch stays 16, so the schedule is unchanged.")
    print("=" * 78)
print()
existing = sorted(p.name for p in CHECKPOINT_DIR.glob("checkpoint-*"))
if existing:
    print("Checkpoints already in Drive from a previous session:", existing)
    print("To resume, set CFG.resume_from_checkpoint to one of:")
    for name in existing:
        print("   ", CHECKPOINT_DIR / name)
for key, value in asdict(CFG).items():
    print(f"  {key} = {value}")

Memory profile: GPU_16GB | REHEARSAL: False
Run root (Drive): /content/drive/MyDrive/bot-training/training_runs/fastAPI_Model-runtime-task

FULL RUN  |  training budget 240 min
  corpus: no per-stage cap -- every unique record
  up to 3 epochs, seq 1024, batch 2 x accum 8 = 16 effective
  checkpoint + eval every 40 steps; best checkpoint by eval_loss is what gets exported
  Trainer eval capped at 256 rows (generation eval uses 90)
  merge -> GGUF f16 -> Q4_K_M -> Drive export bundle: ENABLED

  At the rehearsal's ~40 s/step this is roughly one pass over the whole
  corpus. One pass over every record beats several passes over a slice:
  repeats lower training loss by memorising, coverage is what generalises.

  Expect roughly:
    240 min training
    ~5 min base-model download
    ~22 min generation evaluation (90 records)
    ~8 min merge to fp16
    ~8 min GGUF f16 conversion
    ~5 min llama-quantize build and Q4_K_M quantization
    ~6 min copying the 1.9 GB artifact to Drive
    t

### 3.1 Locate and verify the data


In [9]:
import json  # noqa: E402


def find_data_file(name: str) -> Path | None:
    """Look in the nested layout, the flat layout, then anywhere under the root."""
    for candidate in (DRIVE_ROOT / "training_data" / name, DRIVE_ROOT / name):
        if candidate.exists():
            return candidate
    return next((match for match in DRIVE_ROOT.rglob(name) if match.is_file()), None)


# training_protocol.json is 806 bytes; carrying a copy means a forgotten upload
# does not end the session.
PROTOCOL_FALLBACK = {
    "version": "runtime-task.v1",
    "runtime_contract": "project-run.v1",
    "context_window": 8192,
    "tasks": {
        "schema_plan": {"output": "json", "weight": 15},
        "api_contract": {"output": "json", "weight": 15},
        "file_plan": {"output": "json", "weight": 10},
        "create_file": {"output": "json", "weight": 25},
        "edit_patch": {"output": "json", "weight": 25},
        "validation_repair": {"output": "json", "weight": 10},
    },
    "rules": [
        "Return only the format requested by the stage system message.",
        "Preserve files and behavior outside the explicit change scope.",
        "Never use placeholders, TODOs, or omitted code.",
        "Treat syntax and runtime failures as hard gates.",
        "Treat incomplete inferred artifacts as repair guidance, not permission to erase prior work.",
    ],
}

# The six hand-written runtime-protocol seed records, verbatim from
# bot-training/training_data/runtime_protocol_seed.jsonl. Embedded because the
# file is 5 KB and is the one input most easily forgotten.
SEED_FALLBACK_JSONL = r'''
{"messages":[{"role":"system","content":"Stage: schema_plan. Return only JSON with domain, ddl_sql, schema, assumptions."},{"role":"user","content":"Create a library API with books and authors. Books have title and author_id."},{"role":"assistant","content":"{\"domain\":\"library\",\"ddl_sql\":\"CREATE TABLE authors (id INTEGER PRIMARY KEY, name VARCHAR(160) NOT NULL); CREATE TABLE books (id INTEGER PRIMARY KEY, title VARCHAR(255) NOT NULL, author_id INTEGER NOT NULL REFERENCES authors(id));\",\"schema\":{\"tables\":[{\"name\":\"authors\",\"columns\":[{\"name\":\"id\",\"type\":\"INTEGER\"},{\"name\":\"name\",\"type\":\"VARCHAR(160)\"}]},{\"name\":\"books\",\"columns\":[{\"name\":\"id\",\"type\":\"INTEGER\"},{\"name\":\"title\",\"type\":\"VARCHAR(255)\"},{\"name\":\"author_id\",\"type\":\"INTEGER\"}],\"relationships\":[{\"field\":\"author_id\",\"to\":\"authors.id\"}]}]},\"assumptions\":[]}"}],"_meta":{"task":"schema_plan","protocol":"runtime-task.v1"}}
{"messages":[{"role":"system","content":"Stage: api_contract. Return only JSON with summary, requirements, routes, auth, websocket_events, artifact_contract, assumptions."},{"role":"user","content":"Expose create and list book endpoints and filter books by title."},{"role":"assistant","content":"{\"summary\":\"Book creation and filtered listing\",\"requirements\":[{\"id\":\"R1\",\"description\":\"Create books\"},{\"id\":\"R2\",\"description\":\"List books with an optional title filter\"}],\"routes\":[{\"method\":\"POST\",\"path\":\"/books\"},{\"method\":\"GET\",\"path\":\"/books\"}],\"auth\":null,\"websocket_events\":[],\"artifact_contract\":{\"required_routes\":[{\"method\":\"POST\",\"path\":\"/books\"},{\"method\":\"GET\",\"path\":\"/books\"}],\"required_tables\":[{\"table\":\"books\"}],\"required_fields\":[{\"table\":\"books\",\"field\":\"title\"}],\"required_filters\":[{\"table\":\"books\",\"field\":\"title\"}]},\"assumptions\":[]}"}],"_meta":{"task":"api_contract","protocol":"runtime-task.v1"}}
{"messages":[{"role":"system","content":"Stage: file_plan. Return only JSON with files and notes."},{"role":"user","content":"Plan the files for the supplied FastAPI book contract."},{"role":"assistant","content":"{\"files\":[{\"path\":\"database.py\",\"purpose\":\"Engine and session dependency\",\"depends_on\":[]},{\"path\":\"models.py\",\"purpose\":\"SQLModel tables\",\"depends_on\":[\"database.py\"]},{\"path\":\"schemas.py\",\"purpose\":\"Request and response schemas\",\"depends_on\":[]},{\"path\":\"routers/books.py\",\"purpose\":\"Book routes\",\"depends_on\":[\"models.py\",\"schemas.py\",\"database.py\"]},{\"path\":\"main.py\",\"purpose\":\"Application entrypoint\",\"depends_on\":[\"routers/books.py\"]},{\"path\":\"requirements.txt\",\"purpose\":\"Runtime dependencies\",\"depends_on\":[]}],\"notes\":[]}"}],"_meta":{"task":"file_plan","protocol":"runtime-task.v1"}}
{"messages":[{"role":"system","content":"Stage: create_file. Return only JSON with path and content. Produce complete code without markdown."},{"role":"user","content":"Create schemas.py for BookCreate(title: str, author_id: int) and BookRead with an integer id."},{"role":"assistant","content":"{\"path\":\"schemas.py\",\"content\":\"from pydantic import BaseModel, ConfigDict\\n\\n\\nclass BookCreate(BaseModel):\\n    title: str\\n    author_id: int\\n\\n\\nclass BookRead(BookCreate):\\n    model_config = ConfigDict(from_attributes=True)\\n\\n    id: int\\n\"}"}],"_meta":{"task":"create_file","protocol":"runtime-task.v1"}}
{"messages":[{"role":"system","content":"Stage: edit_patch. Return only JSON with patches. Preserve unrelated code and use exact search text."},{"role":"user","content":"Add an optional title query filter to the existing list_books route. Existing file: routers/books.py contains statement = select(Book) followed by return session.exec(statement).all()."},{"role":"assistant","content":"{\"patches\":[{\"path\":\"routers/books.py\",\"search\":\"def list_books(session: Session = Depends(get_session)):\\n    statement = select(Book)\\n    return session.exec(statement).all()\",\"replace\":\"def list_books(title: str | None = None, session: Session = Depends(get_session)):\\n    statement = select(Book)\\n    if title is not None:\\n        statement = statement.where(Book.title.contains(title))\\n    return session.exec(statement).all()\"}],\"notes\":[]}"}],"_meta":{"task":"edit_patch","protocol":"runtime-task.v1"}}
{"messages":[{"role":"system","content":"Stage: validation_repair. Return only JSON with patches. Repair only the reported failure."},{"role":"user","content":"Validation error in main.py: NameError: books_router is not defined. The file imports from routers import books and calls app.include_router(books_router)."},{"role":"assistant","content":"{\"patches\":[{\"path\":\"main.py\",\"search\":\"app.include_router(books_router)\",\"replace\":\"app.include_router(books.router)\"}],\"notes\":[\"Use the already imported books module; no other behavior changes.\"]}"}],"_meta":{"task":"validation_repair","protocol":"runtime-task.v1"}}
'''

In [10]:
# --- Required inputs -------------------------------------------------------
LEGACY_TRAIN_JSONL = find_data_file("fastapi_directive_dataset.jsonl")
LEGACY_EVAL_JSONL = find_data_file("fastapi_directive_eval.jsonl")

required_missing = [
    name
    for name, path in (
        ("fastapi_directive_dataset.jsonl", LEGACY_TRAIN_JSONL),
        ("fastapi_directive_eval.jsonl", LEGACY_EVAL_JSONL),
    )
    if path is None
]
if required_missing:
    raise FileNotFoundError(
        "Required file(s) not found anywhere under "
        f"{DRIVE_ROOT}: {required_missing}. "
        "Upload them from the local bot-training/training_data folder, then re-run this cell. "
        f"Currently visible: {sorted(p.name for p in DRIVE_ROOT.rglob('*') if p.is_file())[:20]}"
    )

# --- Optional inputs, with embedded fallbacks ------------------------------
SEED_JSONL = find_data_file("runtime_protocol_seed.jsonl")
if SEED_JSONL is None:
    SEED_JSONL = DRIVE_ROOT / "runtime_protocol_seed.jsonl"
    SEED_JSONL.write_text(SEED_FALLBACK_JSONL.strip() + chr(10), encoding="utf-8")
    print(f"runtime_protocol_seed.jsonl was not on Drive; wrote the embedded copy to {SEED_JSONL}")

PROTOCOL_JSON = find_data_file("training_protocol.json")
if PROTOCOL_JSON is None:
    PROTOCOL_JSON = DRIVE_ROOT / "training_protocol.json"
    PROTOCOL_JSON.write_text(json.dumps(PROTOCOL_FALLBACK, indent=2) + chr(10), encoding="utf-8")
    print(f"training_protocol.json was not on Drive; wrote the embedded copy to {PROTOCOL_JSON}")

print()
print("Resolved inputs:")
for path in (LEGACY_TRAIN_JSONL, LEGACY_EVAL_JSONL, SEED_JSONL, PROTOCOL_JSON):
    with path.open("r", encoding="utf-8") as handle:
        rows = sum(1 for line in handle if line.strip())
    where = path.parent.relative_to(DRIVE_ROOT) if path.parent != DRIVE_ROOT else Path(".")
    print(f"  {path.name:<38} {path.stat().st_size:>10,} B  {rows:>5} lines   in ./{where}")

# Sanity-check the seed before the converter relies on it.
seed_tasks = [
    json.loads(line)["_meta"]["task"]
    for line in SEED_JSONL.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print("  seed stages:", seed_tasks)
if len(seed_tasks) != 6:
    print(f"  NOTE: expected 6 seed records, found {len(seed_tasks)}.")

# Copy the two large files to local disk; the converter reads them repeatedly
# and Drive FUSE reads are slow.
_local = Path("/content/staged_data")
_local.mkdir(parents=True, exist_ok=True)
for source in (LEGACY_TRAIN_JSONL, LEGACY_EVAL_JSONL, SEED_JSONL):
    target = _local / source.name
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy2(source, target)
LEGACY_TRAIN_JSONL = _local / LEGACY_TRAIN_JSONL.name
LEGACY_EVAL_JSONL = _local / LEGACY_EVAL_JSONL.name
SEED_JSONL = _local / SEED_JSONL.name
print()
print("Staged to local disk:", _local)


Resolved inputs:
  fastapi_directive_dataset.jsonl        16,227,539 B   5500 lines   in ./.
  fastapi_directive_eval.jsonl            1,496,245 B    500 lines   in ./.
  runtime_protocol_seed.jsonl                 5,061 B      6 lines   in ./.
  training_protocol.json                        806 B     20 lines   in ./.
  seed stages: ['schema_plan', 'api_contract', 'file_plan', 'create_file', 'edit_patch', 'validation_repair']

Staged to local disk: /content/staged_data


## 4. The stage contracts the backend actually sends


In [11]:
import collections  # noqa: E402
import hashlib  # noqa: E402
import json  # noqa: E402
import random  # noqa: E402
import re  # noqa: E402
from typing import Any  # noqa: E402

PROTOCOL = json.loads(PROTOCOL_JSON.read_text(encoding="utf-8-sig"))
TASK_WEIGHTS = {name: spec["weight"] for name, spec in PROTOCOL["tasks"].items()}
print("Protocol:", PROTOCOL["version"], "runtime:", PROTOCOL["runtime_contract"])
print("Task weights:", TASK_WEIGHTS)

# Verbatim from app/services/model_pipeline/profiles.py
ENTRYPOINT_FILE = "main.py"
DATABASE_FILE = "database.py"
DEPENDENCY_FILE = "requirements.txt"
REQUIRED_PROJECT_FILES = {"main.py", "database.py", "requirements.txt"}

# Verbatim from app/services/model_pipeline/artifacts.py
ARTIFACT_KEYS = (
    "required_routes",
    "required_tables",
    "required_fields",
    "required_filters",
    "required_relationships",
    "required_behaviors",
    "removed_artifacts",
)

# Verbatim from editing.py patch_project
ALLOWED_PATCH_OPS = {
    "replace_text",
    "insert_before",
    "insert_after",
    "delete_text",
    "create_file",
    "insert_import",
    "insert_class_field",
    "insert_router_wiring",
    "append_function",
}

STAGE_SYSTEM_PROMPTS: dict[str, str] = {
    "schema_plan": (
        "You are a backend data architect. Produce only the database/schema plan for the requested FastAPI backend. "
        "Do not write application code. Do not choose a fixed template. Do not add features the user did not request. "
        "Return ONLY JSON with keys: domain, ddl_sql, schema, assumptions. "
        "schema.tables must include table name, columns with type/nullability/default when known, relationships, "
        "indexes, and filters/search fields requested by the user."
    ),
    "api_contract": (
        "You are a FastAPI API contract planner. Use the saved schema plan as source context. "
        "Do not write code. Do not invent fixed template APIs. Return ONLY JSON with keys: summary, requirements, "
        "routes, auth, websocket_events, artifact_contract, assumptions. "
        "artifact_contract must explicitly list routes, tables, fields, filters, relationships, and behaviors that prove completion."
    ),
    "file_plan": (
        "You are planning a maintainable FastAPI project file structure. Do not write code. "
        "Use the schema and API contract as the source of truth. Return ONLY JSON with keys: files, notes. "
        "files must be a list of objects with path, purpose, depends_on, related_tables, related_routes. "
        "Include the runnable entry/database/dependency files needed for the project. "
        f"The required entrypoint is {ENTRYPOINT_FILE}; do not use root app.py as an extra app file. "
        "Respect normal FastAPI file roles: database setup belongs in the database file, app setup belongs in the entrypoint, "
        "database models belong in model files, and route handlers belong in router or entrypoint files."
    ),
    "edit_patch": (
        "Return ONLY one JSON object for a FastAPI patch edit. No prose. "
        "Use only source_files and project_index. Do not invent a fixed template. "
        "Required top-level keys: summary, requirements, artifact_contract, target_files, patches, missing_info. "
        "requirements must be non-empty. artifact_contract must explicitly list the routes/tables/fields/relationships "
        "that prove the edit. patches must be non-empty. "
        "Allowed patch ops: replace_text(op,path,old,new), insert_before(op,path,anchor,content), "
        "insert_after(op,path,anchor,content), delete_text(op,path,old), create_file(op,path,content), "
        "insert_import(op,path,statement), insert_class_field(op,path,class_name,content), "
        "insert_router_wiring(op,path,import_statement,include_statement), append_function(op,path,content). "
        "Patch anchors must be exact unique text from source_files."
    ),
    # editing.py repair_validated_edit. {repair_task} is interpolated per call by
    # the backend; training uses the generic objective it falls back to.
    "validation_repair": (
        "The previous FastAPI edit applied to a temporary project but failed one validation group. "
        "Return ONLY one JSON object for a small focused full-file edit. No prose. "
        "Required top-level keys: summary, requirements, artifact_contract, files, missing_info. "
        "files must contain complete updated source only for files needed by repair_task. "
        "Focus only on this repair task: Fix the next validation failure group. "
        "Do not broaden the edit, do not rewrite unrelated files, and do not try to solve unrelated validation groups. "
        "Do not return a patch; return full files for the focused files."
    ),
}

# Verbatim from generation.py FILE_ROLE_RULES
FILE_ROLE_RULES = [
    f"{ENTRYPOINT_FILE}: instantiate exactly one FastAPI app and wire routers/routes.",
    f"{DATABASE_FILE}: database engine/session setup only; no FastAPI app and no route handlers.",
    "model files: define persistence models using SQLModel(table=True) or SQLAlchemy declarative models when database tables are required.",
    "schema files: define request/response Pydantic schemas only; do not replace database models with schemas.",
    "router files: define APIRouter endpoint handlers; do not instantiate FastAPI apps.",
    "auth files: define password/JWT helpers and dependencies; do not instantiate FastAPI apps or duplicate resource routers.",
    "All generated functions must contain real implementation code, not pass/TODO/placeholder comments.",
    "Every name used in route decorators, response_model, dependencies, type annotations, and defaults must be imported or defined.",
]


def create_file_system_prompt(path: str, role_instruction: str) -> str:
    return (
        "You are writing exactly one file for a FastAPI backend. Use the saved schema plan, API contract, "
        "overall file plan, and summaries of already-written files. Do not write unrelated files. "
        "Do not summarize, explain, or describe the plan. Output only complete file content inside this wrapper:\n"
        f"### FILE: {path} ###\n<complete file content>\n### END FILE ###"
        f"\nTarget role: {role_instruction}"
    )


# generation.py repair branch: whole-project repair, wrapper output. Not a
# training target in this notebook; kept so the two repair paths stay documented.
GENERATION_PROJECT_REPAIR_SYSTEM = (
    "You are repairing a generated FastAPI backend that failed artifact validation. "
    "Return a corrected complete project using only file wrappers. Do not output prose. "
    "Treat static file-role errors, unresolved imports, undefined names, placeholder functions, and missing artifacts as blockers. "
    "Implement the missing routes, SQLModel tables, fields, query filters, relationships, JWT auth, "
    "and auth protection described by validation.artifact_validation. "
    f"You may keep the implementation compact in {ENTRYPOINT_FILE} if that is the most reliable fix. "
    f"Include {', '.join(sorted(REQUIRED_PROJECT_FILES))}. "
    "Do not put route handlers in database files. Do not instantiate FastAPI outside the entrypoint. "
    "Do not use Pydantic schemas as substitutes for persistence models when database tables are required. "
    "Use this wrapper for each file:\n### FILE: path/to/file.py ###\n<complete file content>\n### END FILE ###"
)

REQUIRED_KEYS_BY_TASK: dict[str, tuple[str, ...]] = {
    "schema_plan": ("domain", "ddl_sql", "schema", "assumptions"),
    "api_contract": (
        "summary",
        "requirements",
        "routes",
        "auth",
        "websocket_events",
        "artifact_contract",
        "assumptions",
    ),
    "file_plan": ("files", "notes"),
    "edit_patch": (
        "summary",
        "requirements",
        "artifact_contract",
        "target_files",
        "patches",
        "missing_info",
    ),
    "validation_repair": (
        "summary",
        "requirements",
        "artifact_contract",
        "files",
        "missing_info",
    ),
}
# Only create_file answers are free text; every other stage returns one JSON object.
TEXT_TASKS = {"create_file"}
ALL_TASKS = tuple(PROTOCOL["tasks"].keys())

FILE_WRAPPER_RE = re.compile(
    r"###\s*FILE:\s*(?P<path>[^#\n]+?)\s*###\s*\n(?P<body>.*?)\n?###\s*END FILE\s*###",
    re.DOTALL,
)

Protocol: runtime-task.v1 runtime: project-run.v1
Task weights: {'schema_plan': 15, 'api_contract': 15, 'file_plan': 10, 'create_file': 25, 'edit_patch': 25, 'validation_repair': 10}


## 5. JSON, hashing, and structural-signature helpers

In [12]:
def canonical_json(value: Any, *, sort_keys: bool = False) -> str:
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"), sort_keys=sort_keys)


def stable_hash(value: Any) -> str:
    return hashlib.sha256(canonical_json(value, sort_keys=True).encode("utf-8")).hexdigest()


def text_hash(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def extract_json_object(text: str) -> str:
    """Pull the first balanced JSON object out of a model response."""
    raw = str(text or "").strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
        raw = re.sub(r"\s*```$", "", raw)
    start = raw.find("{")
    if start < 0:
        raise ValueError("no JSON object start found")
    depth = 0
    in_string = False
    escape = False
    for idx, char in enumerate(raw[start:], start=start):
        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
            continue
        if char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return raw[start : idx + 1]
    raise ValueError("no complete JSON object found")


def validate_task_payload(task: str, payload: Any, where: str) -> None:
    """Enforce the same required-key contract the backend enforces at runtime."""
    if task in TEXT_TASKS:
        if not isinstance(payload, str) or not FILE_WRAPPER_RE.search(payload):
            raise ValueError(f"{where}: {task} answer has no '### FILE: ... ###' wrapper")
        for match in FILE_WRAPPER_RE.finditer(payload):
            if not match.group("body").strip():
                raise ValueError(f"{where}: {task} wrapper for {match.group('path')} is empty")
        return

    if not isinstance(payload, dict):
        raise ValueError(f"{where}: {task} answer is not a JSON object")
    missing = [key for key in REQUIRED_KEYS_BY_TASK[task] if key not in payload]
    if missing:
        raise ValueError(f"{where}: {task} missing required keys {missing}")

    if task == "schema_plan":
        schema = payload.get("schema")
        if not isinstance(schema, dict) or not schema.get("tables"):
            if not schema or not schema.get("no_database_required"):
                raise ValueError(f"{where}: schema_plan has neither tables nor no_database_required")
        if not str(payload.get("ddl_sql") or "").strip() and schema.get("tables"):
            raise ValueError(f"{where}: schema_plan declares tables but ddl_sql is empty")

    if task == "api_contract":
        contract = payload.get("artifact_contract")
        if not isinstance(contract, dict):
            raise ValueError(f"{where}: api_contract.artifact_contract is not an object")
        unknown = set(contract) - set(ARTIFACT_KEYS)
        if unknown:
            raise ValueError(f"{where}: artifact_contract has unknown keys {sorted(unknown)}")
        if not payload.get("routes"):
            raise ValueError(f"{where}: api_contract.routes is empty")

    if task == "file_plan":
        paths = {str(item.get("path") or "") for item in payload.get("files") or []}
        missing_files = sorted(REQUIRED_PROJECT_FILES - paths)
        if missing_files:
            raise ValueError(f"{where}: file_plan omits required files {missing_files}")

    if task == "edit_patch":
        patches = payload.get("patches") or []
        if not patches:
            raise ValueError(f"{where}: edit_patch.patches is empty")
        for index, patch in enumerate(patches):
            op = patch.get("op")
            if op not in ALLOWED_PATCH_OPS:
                raise ValueError(f"{where}: patch {index} uses unsupported op {op!r}")
            if not patch.get("path"):
                raise ValueError(f"{where}: patch {index} has no path")

    if task == "validation_repair":
        if payload.get("patches"):
            raise ValueError(
                f"{where}: validation_repair returned patches; the stage requires complete files"
            )
        files = payload.get("files") or []
        if not files:
            raise ValueError(f"{where}: validation_repair.files is empty")
        for index, item in enumerate(files):
            if not str(item.get("path") or "").strip():
                raise ValueError(f"{where}: repair file {index} has no path")
            if not str(item.get("content") or "").strip():
                raise ValueError(f"{where}: repair file {index} has no content")


def structural_signature(task: str, payload: Any) -> dict[str, set[str]]:
    """Comparable sets per stage, used for component F1 in section 11."""
    sig: dict[str, set[str]] = collections.defaultdict(set)
    if task == "schema_plan" and isinstance(payload, dict):
        schema = payload.get("schema") or {}
        for table in schema.get("tables") or []:
            name = str(table.get("name") or "").strip()
            if not name:
                continue
            sig["tables"].add(name)
            for column in table.get("columns") or []:
                column_name = str(column.get("name") or "").strip()
                if column_name:
                    sig["columns"].add(f"{name}.{column_name}")
            for relationship in table.get("relationships") or []:
                sig["relationships"].add(
                    f"{name}.{relationship.get('field')}->{relationship.get('to')}"
                )
            for filter_field in table.get("filters") or []:
                sig["filters"].add(f"{name}.{filter_field}")
    elif task == "api_contract" and isinstance(payload, dict):
        for route in payload.get("routes") or []:
            sig["routes"].add(f"{str(route.get('method') or '').upper()} {route.get('path')}")
        contract = payload.get("artifact_contract") or {}
        for route in contract.get("required_routes") or []:
            sig["required_routes"].add(
                f"{str(route.get('method') or '').upper()} {route.get('path')}"
            )
        for table in contract.get("required_tables") or []:
            sig["required_tables"].add(str(table.get("table") or table))
        for item in contract.get("required_fields") or []:
            sig["required_fields"].add(f"{item.get('table')}.{item.get('field')}")
        for item in contract.get("required_filters") or []:
            sig["required_filters"].add(f"{item.get('table')}.{item.get('field')}")
        sig["auth"].add("jwt" if payload.get("auth") else "none")
    elif task == "file_plan" and isinstance(payload, dict):
        for item in payload.get("files") or []:
            path = str(item.get("path") or "").strip()
            if path:
                sig["files"].add(path)
    elif task == "edit_patch" and isinstance(payload, dict):
        for patch in payload.get("patches") or []:
            sig["patch_ops"].add(f"{patch.get('op')}:{patch.get('path')}")
        for path in payload.get("target_files") or []:
            sig["target_files"].add(str(path))
    elif task == "validation_repair" and isinstance(payload, dict):
        for item in payload.get("files") or []:
            path = str(item.get("path") or "").strip()
            if path:
                sig["files"].add(path)
    elif isinstance(payload, str):
        for match in FILE_WRAPPER_RE.finditer(payload):
            sig["files"].add(match.group("path").strip())
    return dict(sig)


def f1(pred: set[str], gold: set[str]) -> float:
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    true_positive = len(pred & gold)
    if not true_positive:
        return 0.0
    precision = true_positive / len(pred)
    recall = true_positive / len(gold)
    return 2 * precision * recall / (precision + recall)

## 6. Build the `runtime-task.v1` corpus


In [13]:
FIELD_TYPE_TO_SQL = {
    "str": "VARCHAR(255)",
    "text": "TEXT",
    "int": "INTEGER",
    "float": "FLOAT",
    "bool": "BOOLEAN",
    "datetime": "DATETIME",
    "date": "DATE",
    "enum": "VARCHAR(64)",
    "fk": "INTEGER",
    "email": "VARCHAR(320)",
}
OP_TO_ROUTE = {
    "create": ("POST", ""),
    "list": ("GET", ""),
    "retrieve": ("GET", "/{item_id}"),
    "update": ("PUT", "/{item_id}"),
    "delete": ("DELETE", "/{item_id}"),
}


def to_snake(name: str) -> str:
    text = re.sub(r"(?<!^)(?=[A-Z])", "_", str(name).strip()).lower()
    return re.sub(r"[^a-z0-9_]+", "_", text).strip("_") or "resource"


def pluralize(name: str) -> str:
    if name.endswith(("s", "x", "z", "ch", "sh")):
        return name + "es"
    if name.endswith("y") and not name.endswith(("ay", "ey", "iy", "oy", "uy")):
        return name[:-1] + "ies"
    return name + "s"


def table_name_for(resource: str) -> str:
    return pluralize(to_snake(resource))


def _resources_and_packs(plan: dict[str, Any]) -> tuple[list[dict[str, Any]], set[str]]:
    resources: list[dict[str, Any]] = []
    packs: set[str] = set()
    for directive in plan.get("directives") or []:
        action = directive.get("action")
        if action == "attach_pack":
            packs.add(str(directive.get("pack")))
        elif action == "add_resource":
            resources.append(directive)
    return resources, packs


def build_schema_plan(plan: dict[str, Any]) -> dict[str, Any]:
    resources, packs = _resources_and_packs(plan)
    tables: list[dict[str, Any]] = []
    ddl_statements: list[str] = []

    if "auth" in packs:
        tables.append(
            {
                "name": "users",
                "columns": [
                    {"name": "id", "type": "INTEGER", "nullable": False, "primary_key": True},
                    {"name": "email", "type": "VARCHAR(320)", "nullable": False, "unique": True},
                    {"name": "hashed_password", "type": "VARCHAR(255)", "nullable": False},
                    {"name": "is_active", "type": "BOOLEAN", "nullable": False, "default": "true"},
                ],
                "relationships": [],
                "indexes": ["email"],
                "filters": ["email"],
            }
        )
        ddl_statements.append(
            "CREATE TABLE users (id INTEGER PRIMARY KEY, email VARCHAR(320) NOT NULL UNIQUE, "
            "hashed_password VARCHAR(255) NOT NULL, is_active BOOLEAN NOT NULL DEFAULT true);"
        )

    for resource in resources:
        table = table_name_for(resource.get("name") or "")
        columns: list[dict[str, Any]] = [
            {"name": "id", "type": "INTEGER", "nullable": False, "primary_key": True}
        ]
        relationships: list[dict[str, Any]] = []
        indexes: list[str] = []
        ddl_columns = ["id INTEGER PRIMARY KEY"]

        for field_spec in resource.get("fields") or []:
            field_name = to_snake(field_spec.get("name") or "")
            if not field_name:
                continue
            field_type = str(field_spec.get("type") or "str")
            sql_type = FIELD_TYPE_TO_SQL.get(field_type, "VARCHAR(255)")
            if field_type == "str" and field_spec.get("max_length"):
                sql_type = f"VARCHAR({int(field_spec['max_length'])})"
            nullable = bool(field_spec.get("nullable"))
            unique = bool(field_spec.get("unique"))
            column: dict[str, Any] = {
                "name": field_name,
                "type": sql_type,
                "nullable": nullable,
                "unique": unique,
            }
            fragment = f"{field_name} {sql_type}"
            if not nullable:
                fragment += " NOT NULL"
            if unique:
                fragment += " UNIQUE"
            if field_type == "fk" and field_spec.get("to"):
                target = table_name_for(field_spec["to"])
                column["references"] = f"{target}.id"
                fragment += f" REFERENCES {target}(id)"
                relationships.append({"field": field_name, "to": f"{target}.id"})
            columns.append(column)
            ddl_columns.append(fragment)
            if field_spec.get("indexed"):
                indexes.append(field_name)

        filters = [to_snake(f) for f in resource.get("list_filters") or []]
        for filter_field in filters:
            if filter_field not in indexes:
                indexes.append(filter_field)

        tables.append(
            {
                "name": table,
                "columns": columns,
                "relationships": relationships,
                "indexes": indexes,
                "filters": filters,
            }
        )
        ddl_statements.append(f"CREATE TABLE {table} ({', '.join(ddl_columns)});")

    return {
        "domain": plan.get("description") or plan.get("project_name") or "generated backend",
        "ddl_sql": " ".join(ddl_statements),
        "schema": {"tables": tables},
        "assumptions": [],
    }


def build_api_contract(plan: dict[str, Any]) -> dict[str, Any]:
    resources, packs = _resources_and_packs(plan)
    routes: list[dict[str, Any]] = []
    requirements: list[dict[str, Any]] = []
    required_routes: list[dict[str, str]] = []
    required_tables: list[dict[str, str]] = []
    required_fields: list[dict[str, str]] = []
    required_filters: list[dict[str, str]] = []
    required_relationships: list[dict[str, str]] = []
    required_behaviors: list[dict[str, Any]] = []
    protected: list[str] = []
    counter = 0

    if "auth" in packs:
        counter += 1
        requirements.append(
            {"id": f"R{counter}", "description": "Register and authenticate users with JWT tokens"}
        )
        for method, path in (("POST", "/auth/register"), ("POST", "/auth/login")):
            routes.append({"method": method, "path": path})
            required_routes.append({"method": method, "path": path})
        required_tables.append({"table": "users"})
        required_fields.append({"table": "users", "field": "hashed_password"})
        required_behaviors.append({"behavior": "jwt_authentication", "detail": "Issue and verify bearer tokens"})

    for resource in resources:
        table = table_name_for(resource.get("name") or "")
        base = f"/{table}"
        required_tables.append({"table": table})
        operations = list(resource.get("include") or [])
        counter += 1
        requirements.append(
            {
                "id": f"R{counter}",
                "description": f"Expose {', '.join(operations) or 'read'} operations for {table}",
            }
        )
        for operation in operations:
            if operation not in OP_TO_ROUTE:
                continue
            method, suffix = OP_TO_ROUTE[operation]
            path = base + suffix
            routes.append({"method": method, "path": path})
            required_routes.append({"method": method, "path": path})
        if resource.get("auth_required"):
            protected.append(base)
            required_behaviors.append({"behavior": "auth_protected_routes", "detail": base})
        for field_spec in resource.get("fields") or []:
            field_name = to_snake(field_spec.get("name") or "")
            if not field_name:
                continue
            required_fields.append({"table": table, "field": field_name})
            if field_spec.get("type") == "fk" and field_spec.get("to"):
                required_relationships.append(
                    {
                        "table": table,
                        "field": field_name,
                        "references": f"{table_name_for(field_spec['to'])}.id",
                    }
                )
        for filter_field in resource.get("list_filters") or []:
            filter_name = to_snake(filter_field)
            required_filters.append({"table": table, "field": filter_name})
            counter += 1
            requirements.append(
                {"id": f"R{counter}", "description": f"Filter {table} list results by {filter_name}"}
            )

    websocket_events: list[dict[str, Any]] = []
    if "websocket" in packs:
        websocket_events = [
            {"event": f"{table_name_for(resource.get('name') or '')}.changed", "channel": "updates"}
            for resource in resources
        ]
        required_behaviors.append({"behavior": "websocket_broadcast", "detail": "updates channel"})
    if "files" in packs:
        for method, path in (("POST", "/files/upload"), ("GET", "/files/{file_id}")):
            routes.append({"method": method, "path": path})
            required_routes.append({"method": method, "path": path})
        required_behaviors.append({"behavior": "file_upload_download", "detail": "multipart upload and streamed download"})

    return {
        "summary": plan.get("description") or "generated FastAPI backend",
        "requirements": requirements,
        "routes": routes,
        "auth": {"scheme": "jwt", "protected_prefixes": protected} if "auth" in packs else None,
        "websocket_events": websocket_events,
        "artifact_contract": {
            "required_routes": required_routes,
            "required_tables": required_tables,
            "required_fields": required_fields,
            "required_filters": required_filters,
            "required_relationships": required_relationships,
            "required_behaviors": required_behaviors,
            "removed_artifacts": [],
        },
        "assumptions": [],
    }


def build_file_plan(plan: dict[str, Any]) -> dict[str, Any]:
    resources, packs = _resources_and_packs(plan)
    tables = [table_name_for(r.get("name") or "") for r in resources]
    files: list[dict[str, Any]] = [
        {
            "path": DATABASE_FILE,
            "purpose": "SQLModel engine and session dependency",
            "depends_on": [],
            "related_tables": [],
            "related_routes": [],
        },
        {
            "path": "models.py",
            "purpose": "SQLModel table definitions",
            "depends_on": [DATABASE_FILE],
            "related_tables": (["users"] if "auth" in packs else []) + tables,
            "related_routes": [],
        },
        {
            "path": "schemas.py",
            "purpose": "Pydantic request and response schemas",
            "depends_on": [],
            "related_tables": tables,
            "related_routes": [],
        },
    ]
    if "auth" in packs:
        files.append(
            {
                "path": "auth.py",
                "purpose": "Password hashing, JWT helpers, and the current-user dependency",
                "depends_on": ["models.py", DATABASE_FILE],
                "related_tables": ["users"],
                "related_routes": ["POST /auth/register", "POST /auth/login"],
            }
        )
    for resource, table in zip(resources, tables):
        operations = list(resource.get("include") or [])
        files.append(
            {
                "path": f"routers/{table}.py",
                "purpose": f"APIRouter endpoints for {table}",
                "depends_on": ["models.py", "schemas.py", DATABASE_FILE]
                + (["auth.py"] if resource.get("auth_required") else []),
                "related_tables": [table],
                "related_routes": [
                    f"{OP_TO_ROUTE[op][0]} /{table}{OP_TO_ROUTE[op][1]}"
                    for op in operations
                    if op in OP_TO_ROUTE
                ],
            }
        )
    files.append(
        {
            "path": ENTRYPOINT_FILE,
            "purpose": "FastAPI application entrypoint and router wiring",
            "depends_on": [f"routers/{table}.py" for table in tables] or [DATABASE_FILE],
            "related_tables": tables,
            "related_routes": [],
        }
    )
    files.append(
        {
            "path": DEPENDENCY_FILE,
            "purpose": "Runtime dependencies",
            "depends_on": [],
            "related_tables": [],
            "related_routes": [],
        }
    )
    return {"files": files, "notes": []}


CONVERTERS = {
    "schema_plan": build_schema_plan,
    "api_contract": build_api_contract,
    "file_plan": build_file_plan,
}

In [14]:
def make_record(task: str, system: str, user: str, answer: Any, source: str) -> dict[str, Any]:
    content = answer if isinstance(answer, str) else canonical_json(answer)
    return {
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": content},
        ],
        "task": task,
        "source": source,
        "prompt_hash": text_hash(user),
        "answer_hash": text_hash(content),
    }


class StaleSeedRecord(Exception):
    """A seed answer whose shape cannot be lifted to the current contract."""


def normalize_seed_answer(task: str, content: str) -> Any:
    """Bring the shipped seed answers up to the format the backend parses today."""
    if task in TEXT_TASKS:
        if FILE_WRAPPER_RE.search(content):
            return content
        payload = json.loads(extract_json_object(content))
        path = str(payload.get("path") or "").strip()
        body = str(payload.get("content") or "")
        if not path or not body:
            raise ValueError(f"seed {task} record has no usable path/content")
        return f"### FILE: {path} ###\n{body.rstrip()}\n### END FILE ###"

    payload = json.loads(extract_json_object(content))

    if task == "validation_repair" and payload.get("patches") and not payload.get("files"):
        raise StaleSeedRecord(
            "the seed validation_repair answer is patch-shaped, but the stage requires "
            "complete file bodies that the seed does not contain"
        )

    if task == "edit_patch":
        patches = []
        for patch in payload.get("patches") or []:
            if "op" in patch:
                patches.append(patch)
                continue
            patches.append(
                {
                    "op": "replace_text",
                    "path": patch.get("path"),
                    "old": patch.get("search"),
                    "new": patch.get("replace"),
                }
            )
        payload["patches"] = patches
        payload.setdefault("summary", "Apply the requested scoped edit")
        payload.setdefault(
            "requirements",
            [{"id": "R1", "description": payload.get("summary") or "Scoped edit"}],
        )
        payload.setdefault("artifact_contract", {key: [] for key in ARTIFACT_KEYS})
        payload.setdefault(
            "target_files",
            sorted({str(p.get("path")) for p in patches if p.get("path")}),
        )
        payload.setdefault("missing_info", [])
        payload.pop("notes", None)

    return payload


SEED_REJECTIONS: list[dict[str, str]] = []


def load_seed_records() -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with SEED_JSONL.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            task = (row.get("_meta") or {}).get("task")
            if task not in ALL_TASKS:
                raise ValueError(f"seed:{line_no}: unknown task {task!r}")
            messages = row["messages"]
            user = next(m["content"] for m in messages if m["role"] == "user")
            raw_answer = next(m["content"] for m in reversed(messages) if m["role"] == "assistant")
            try:
                answer = normalize_seed_answer(task, raw_answer)
                validate_task_payload(task, answer, f"seed:{line_no}")
            except (StaleSeedRecord, ValueError) as exc:
                SEED_REJECTIONS.append({"task": task, "line": str(line_no), "reason": str(exc)})
                print(f"  DROP seed:{line_no} [{task}] {exc}")
                continue
            if task == "create_file":
                first = FILE_WRAPPER_RE.search(answer)
                system = create_file_system_prompt(
                    first.group("path").strip(),
                    "Follow the FastAPI file-role rules for this path.",
                )
            else:
                system = STAGE_SYSTEM_PROMPTS[task]
            records.append(make_record(task, system, user, answer, "runtime_protocol_seed"))
    return records


def load_converted_legacy(path: Path, split_name: str) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    stats = collections.Counter()
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            plan = row.get("gold_plan")
            if plan is None:
                assistant = next(
                    m["content"] for m in reversed(row["messages"]) if m["role"] == "assistant"
                )
                plan = json.loads(extract_json_object(assistant))
            prompt = row.get("prompt") or next(
                m["content"] for m in row["messages"] if m["role"] == "user"
            )
            resources, _packs = _resources_and_packs(plan)
            if not resources:
                stats["skipped_no_resources"] += 1
                continue
            for task, converter in CONVERTERS.items():
                try:
                    answer = converter(plan)
                    validate_task_payload(task, answer, f"{split_name}:{line_no}:{task}")
                except Exception as exc:  # noqa: BLE001
                    stats[f"rejected_{task}"] += 1
                    if stats[f"rejected_{task}"] <= 3:
                        print(f"  reject {split_name}:{line_no} {task}: {exc}")
                    continue
                records.append(
                    make_record(
                        task,
                        STAGE_SYSTEM_PROMPTS[task],
                        prompt,
                        answer,
                        f"legacy_directive:{split_name}",
                    )
                )
                stats[f"built_{task}"] += 1
    print(f"{split_name}:", dict(stats))
    return records


print("Loading the runtime protocol seed...")
seed_records = load_seed_records() if CFG.use_runtime_seed else []
print("  seed records:", len(seed_records))

converted_train: list[dict[str, Any]] = []
converted_eval: list[dict[str, Any]] = []
if CFG.use_converted_legacy:
    print("Converting the legacy ProjectPlan corpus into runtime-task records...")
    converted_train = load_converted_legacy(LEGACY_TRAIN_JSONL, "legacy_train")
    converted_eval = load_converted_legacy(LEGACY_EVAL_JSONL, "legacy_eval")
print("  converted train:", len(converted_train), " converted eval:", len(converted_eval))

Loading the runtime protocol seed...
  DROP seed:6 [validation_repair] the seed validation_repair answer is patch-shaped, but the stage requires complete file bodies that the seed does not contain
  seed records: 5
Converting the legacy ProjectPlan corpus into runtime-task records...
legacy_train: {'built_schema_plan': 5500, 'built_api_contract': 5500, 'built_file_plan': 5500}
legacy_eval: {'built_schema_plan': 500, 'built_api_contract': 500, 'built_file_plan': 500}
  converted train: 16500  converted eval: 1500


### 6.1 Deduplicate, weight, split, and check leakage

In [15]:
rng = random.Random(CFG.seed)


def deduplicate(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    seen: set[tuple[str, str, str]] = set()
    unique: list[dict[str, Any]] = []
    dropped = 0
    for record in records:
        key = (record["task"], record["prompt_hash"], record["answer_hash"])
        if key in seen:
            dropped += 1
            continue
        seen.add(key)
        unique.append(record)
    print(f"  deduplicate: kept {len(unique)}, dropped {dropped}")
    return unique


print("Deduplicating...")
pool_train = deduplicate(seed_records + converted_train)
pool_eval = deduplicate(converted_eval)

# Hold out an eval slice from the converted training pool as well, because the
# legacy eval file shares its generator with the training file.
by_task_train: dict[str, list[dict[str, Any]]] = collections.defaultdict(list)
for record in pool_train:
    by_task_train[record["task"]].append(record)

holdout: list[dict[str, Any]] = []
kept_train: list[dict[str, Any]] = []
for task, records in by_task_train.items():
    rng.shuffle(records)
    # Seed-only tasks are too small to split; keep every example for training.
    if len(records) < 20:
        kept_train.extend(records)
        continue
    cut = max(1, int(len(records) * CFG.eval_ratio))
    holdout.extend(records[:cut])
    kept_train.extend(records[cut:])

print(f"Held out {len(holdout)} records from the training pool for evaluation.")


ABUNDANT_THRESHOLD = 20


def resample_to_weights(
    records: list[dict[str, Any]], weights: dict[str, int]
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    """Balance the task mix toward the protocol weights without inventing data.

    Balancing is applied only among stages that actually have enough records.
    A stage holding one seed example must not throttle a stage holding 1,592:
    scaling every task down to the starved task's ratio would discard the whole
    corpus to honour a proportion the data cannot support. Starved stages
    contribute everything they have and are reported as starved instead.
    """
    grouped: dict[str, list[dict[str, Any]]] = collections.defaultdict(list)
    for record in records:
        grouped[record["task"]].append(record)

    available = {task: len(rows) for task, rows in grouped.items()}
    abundant = [
        task
        for task, count in available.items()
        if count >= ABUNDANT_THRESHOLD and weights.get(task)
    ]
    starved = [task for task in ALL_TASKS if task not in abundant]
    scale = min((available[task] / weights[task] for task in abundant), default=0.0)

    selected: list[dict[str, Any]] = []
    target: dict[str, int] = {}
    for task, rows in grouped.items():
        rng.shuffle(rows)
        if task in abundant:
            want = min(len(rows), int(round(scale * weights[task])))
        else:
            want = len(rows)  # take everything a starved stage has
        target[task] = want
        selected.extend(rows[:want])

    if CFG.max_records_per_task:
        capped: dict[str, list[dict[str, Any]]] = collections.defaultdict(list)
        for record in selected:
            if len(capped[record["task"]]) < CFG.max_records_per_task:
                capped[record["task"]].append(record)
        selected = [row for rows in capped.values() for row in rows]

    achieved = collections.Counter(record["task"] for record in selected)
    total = sum(achieved.values()) or 1
    report = {
        "available": available,
        "selected": dict(achieved),
        "balanced_stages": sorted(abundant),
        "starved_stages": starved,
        "target_weight_pct": {task: weights.get(task, 0) for task in ALL_TASKS},
        "achieved_pct": {task: round(100 * achieved.get(task, 0) / total, 2) for task in ALL_TASKS},
        "weight_debt_pct": {
            task: round(weights.get(task, 0) - 100 * achieved.get(task, 0) / total, 2)
            for task in ALL_TASKS
        },
    }
    return selected, report


print()
print("Resampling the training mix toward the protocol weights...")
train_records, mix_report = resample_to_weights(kept_train, TASK_WEIGHTS)
eval_records = deduplicate(holdout + pool_eval)

print(json.dumps(mix_report, indent=2, ensure_ascii=False))

Deduplicating...
  deduplicate: kept 5081, dropped 11424
  deduplicate: kept 1116, dropped 384
Held out 303 records from the training pool for evaluation.

Resampling the training mix toward the protocol weights...
  deduplicate: kept 1371, dropped 48
{
  "available": {
    "schema_plan": 1592,
    "api_contract": 1592,
    "file_plan": 1592,
    "create_file": 1,
    "edit_patch": 1
  },
  "selected": {
    "schema_plan": 1592,
    "api_contract": 1592,
    "file_plan": 1061,
    "create_file": 1,
    "edit_patch": 1
  },
  "balanced_stages": [
    "api_contract",
    "file_plan",
    "schema_plan"
  ],
  "starved_stages": [
    "create_file",
    "edit_patch",
    "validation_repair"
  ],
  "target_weight_pct": {
    "schema_plan": 15,
    "api_contract": 15,
    "file_plan": 10,
    "create_file": 25,
    "edit_patch": 25,
    "validation_repair": 10
  },
  "achieved_pct": {
    "schema_plan": 37.49,
    "api_contract": 37.49,
    "file_plan": 24.98,
    "create_file": 0.02,
    "ed

In [16]:
# Leakage: an eval prompt that also appears in training makes every downstream
# number meaningless, so this is a hard stop rather than a warning.
train_prompt_keys = {(r["task"], r["prompt_hash"]) for r in train_records}
leaked = [r for r in eval_records if (r["task"], r["prompt_hash"]) in train_prompt_keys]
eval_records = [r for r in eval_records if (r["task"], r["prompt_hash"]) not in train_prompt_keys]

print("Train records:", len(train_records))
print("Eval records:", len(eval_records))
print("Leaked eval prompts removed:", len(leaked))
if not eval_records:
    raise RuntimeError("Every eval record leaked into training. Rebuild the split.")

thin_tasks = {
    task: count
    for task in ALL_TASKS
    if (count := sum(1 for r in train_records if r["task"] == task))
    < max(1, CFG.require_min_records_per_task)
}
print()
print("Per-task training coverage:")
for task in ALL_TASKS:
    count = sum(1 for r in train_records if r["task"] == task)
    weight = TASK_WEIGHTS.get(task, 0)
    flag = "  <-- THIN" if count < 20 else ""
    print(f"  {task:<20} weight {weight:>3}%   records {count:>6}{flag}")

if CFG.require_min_records_per_task and thin_tasks:
    raise RuntimeError(
        f"require_min_records_per_task={CFG.require_min_records_per_task} not met for {thin_tasks}. "
        "Author more seed records for those stages before training."
    )
if any(sum(1 for r in train_records if r["task"] == task) < 20 for task in ALL_TASKS):
    print()
    print("WARNING: at least one stage is trained on fewer than 20 examples.")
    print("Expect no measurable gain on that stage. Report it as untrained, not as trained.")

rng.shuffle(train_records)
rng.shuffle(eval_records)

TRAIN_JSONL = BUILT_DATA_DIR / "runtime_task_train.jsonl"
EVAL_JSONL = BUILT_DATA_DIR / "runtime_task_eval.jsonl"
for output_path, rows in ((TRAIN_JSONL, train_records), (EVAL_JSONL, eval_records)):
    with output_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(canonical_json(row) + "\n")
    print("Wrote", output_path, f"({len(rows)} rows)")

(REPORT_DIR / "corpus_report.json").write_text(
    json.dumps(
        {
            "protocol": PROTOCOL["version"],
            "mix": mix_report,
            "train_rows": len(train_records),
            "eval_rows": len(eval_records),
            "leaked_removed": len(leaked),
            "seed_records_dropped_as_stale": SEED_REJECTIONS,
            "per_task_train": {
                task: sum(1 for r in train_records if r["task"] == task) for task in ALL_TASKS
            },
            "per_task_eval": {
                task: sum(1 for r in eval_records if r["task"] == task) for task in ALL_TASKS
            },
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

Train records: 4247
Eval records: 642
Leaked eval prompts removed: 729

Per-task training coverage:
  schema_plan          weight  15%   records   1592
  api_contract         weight  15%   records   1592
  file_plan            weight  10%   records   1061
  create_file          weight  25%   records      1  <-- THIN
  edit_patch           weight  25%   records      1  <-- THIN
  validation_repair    weight  10%   records      0  <-- THIN

Expect no measurable gain on that stage. Report it as untrained, not as trained.
Wrote /content/drive/MyDrive/bot-training/training_runs/fastAPI_Model-runtime-task/data/runtime_task_train.jsonl (4247 rows)
Wrote /content/drive/MyDrive/bot-training/training_runs/fastAPI_Model-runtime-task/data/runtime_task_eval.jsonl (642 rows)


1789

### 6.2 Corpus diagnostics

In [17]:
def summarize_corpus(records: list[dict[str, Any]], name: str) -> None:
    char_counts = [sum(len(m["content"]) for m in r["messages"]) for r in records]
    answer_counts = [len(r["messages"][-1]["content"]) for r in records]
    sources = collections.Counter(r["source"] for r in records)
    print(f"== {name} ==")
    print("  rows:", len(records))
    print(
        "  total chars  min/avg/max:",
        min(char_counts),
        round(sum(char_counts) / len(char_counts), 1),
        max(char_counts),
    )
    print(
        "  answer chars min/avg/max:",
        min(answer_counts),
        round(sum(answer_counts) / len(answer_counts), 1),
        max(answer_counts),
    )
    print("  sources:", sources.most_common())


summarize_corpus(train_records, "train")
summarize_corpus(eval_records, "eval")

example = next(r for r in train_records if r["task"] == "api_contract")
print()
print("Example api_contract record")
print("  user:", example["messages"][1]["content"][:220])
print("  assistant:", example["messages"][2]["content"][:400])

== train ==
  rows: 4247
  total chars  min/avg/max: 755 2318.9 4529
  answer chars min/avg/max: 244 1549.7 3758
  sources: [('legacy_directive:legacy_train', 4244), ('runtime_protocol_seed', 3)]
== eval ==
  rows: 642
  total chars  min/avg/max: 1001 2268.5 4351
  answer chars min/avg/max: 457 1457.1 3594
  sources: [('legacy_directive:legacy_eval', 339), ('legacy_directive:legacy_train', 302), ('runtime_protocol_seed', 1)]

Example api_contract record
  user: Create a generated FastAPI project named support_api_2206 for Help desk backend. The API needs tickets, and replies. Resource endpoints can be public. Attach websocket notifications. The generated plan should include lis
  assistant: {"summary":"Help desk backend","requirements":[{"id":"R1","description":"Expose list, retrieve operations for tickets"},{"id":"R2","description":"Filter tickets list results by status"},{"id":"R3","description":"Filter tickets list results by requester_email"},{"id":"R4","description":"Expose list, 

## 7. Tokenize with assistant-only loss

In [18]:
from datasets import Dataset  # noqa: E402
from transformers import AutoTokenizer  # noqa: E402

tokenizer = AutoTokenizer.from_pretrained(CFG.base_model_id, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def render_chat(messages: list[dict[str, str]], *, add_generation_prompt: bool) -> str:
    kwargs = {"tokenize": False, "add_generation_prompt": add_generation_prompt}
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)


def tokenize_assistant_only(example: dict[str, Any]) -> dict[str, Any]:
    messages = example["messages"]
    prompt_messages = [m for m in messages if m["role"] != "assistant"]
    prompt_text = render_chat(prompt_messages, add_generation_prompt=True)
    full_text = render_chat(messages, add_generation_prompt=False)

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

    if len(full_ids) <= len(prompt_ids):
        # Chat template did not append the answer as expected; build it manually.
        answer_ids = tokenizer(
            messages[-1]["content"] + tokenizer.eos_token, add_special_tokens=False
        )["input_ids"]
        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids
    else:
        input_ids = full_ids
        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids) :]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "length": len(input_ids),
        "answer_tokens": sum(1 for label in labels if label != -100),
        "too_long": len(input_ids) > CFG.max_seq_length,
        "task": example["task"],
    }


raw_train = Dataset.from_list(train_records)
raw_eval = Dataset.from_list(eval_records)

tokenized_train = raw_train.map(tokenize_assistant_only, remove_columns=raw_train.column_names)
tokenized_eval = raw_eval.map(tokenize_assistant_only, remove_columns=raw_eval.column_names)

dropped_train_by_task = collections.Counter(
    row["task"] for row in tokenized_train if row["too_long"]
)
dropped_eval_by_task = collections.Counter(row["task"] for row in tokenized_eval if row["too_long"])

train_dataset = tokenized_train.filter(lambda r: not r["too_long"] and r["answer_tokens"] > 0)
eval_dataset = tokenized_eval.filter(lambda r: not r["too_long"] and r["answer_tokens"] > 0)

eval_task_labels = list(eval_dataset["task"])
train_dataset = train_dataset.remove_columns(["too_long", "task"])
eval_dataset = eval_dataset.remove_columns(["too_long", "task"])

lengths = train_dataset["length"]
answer_lengths = train_dataset["answer_tokens"]
print("Train token rows:", len(train_dataset))
print("Eval token rows:", len(eval_dataset))
print("Dropped over-length (train):", dict(dropped_train_by_task))
print("Dropped over-length (eval):", dict(dropped_eval_by_task))
print(
    "Sequence length min/avg/max:",
    min(lengths),
    round(sum(lengths) / len(lengths), 1),
    max(lengths),
)
print("Answer tokens avg:", round(sum(answer_lengths) / len(answer_lengths), 1))
if dropped_train_by_task:
    print()
    print(f"NOTE: max_seq_length={CFG.max_seq_length} is dropping rows.")
    print("On a larger card, raise the memory profile rather than truncating answers.")
if not len(train_dataset):
    raise RuntimeError("Every training row was filtered out. Raise max_seq_length.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/4247 [00:00<?, ? examples/s]

Map:   0%|          | 0/642 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4247 [00:00<?, ? examples/s]

Filter:   0%|          | 0/642 [00:00<?, ? examples/s]

Train token rows: 4247
Eval token rows: 642
Dropped over-length (train): {}
Dropped over-length (eval): {}
Sequence length min/avg/max: 183 524.8 1016
Answer tokens avg: 358.6


## 8. Load the base model in 4-bit and attach LoRA

`nf4` with double quantization, compute dtype chosen by section 2 — fp16 on Turing, bf16 from Ampere on. `use_cache` is disabled because gradient checkpointing and the KV cache cannot both be active.

In [19]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training  # noqa: E402
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, set_seed  # noqa: E402

set_seed(CFG.seed)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    CFG.base_model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
    torch_dtype=COMPUTE_DTYPE,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    target_modules=list(CFG.lora_targets),
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(
    "Allocated / reserved VRAM (GiB):",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    round(torch.cuda.memory_reserved() / 1024**3, 2),
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031
Allocated / reserved VRAM (GiB): 2.72 2.77


## 9. Train


In [20]:
import inspect  # noqa: E402
import math  # noqa: E402
import time  # noqa: E402
from dataclasses import dataclass as _dataclass  # noqa: E402

import transformers  # noqa: E402
from transformers import (  # noqa: E402
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

print("transformers:", transformers.__version__)


@_dataclass
class AssistantOnlyCollator:
    tokenizer: Any
    pad_to_multiple_of: int = 8

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        labels = [feature["labels"] for feature in features]
        inputs = [
            {"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features
        ]
        batch = self.tokenizer.pad(
            inputs,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        padded = torch.full(batch["input_ids"].shape, -100, dtype=torch.long)
        for index, label in enumerate(labels):
            tensor = torch.tensor(label, dtype=torch.long)
            padded[index, : tensor.numel()] = tensor
        batch["labels"] = padded
        return batch


effective_batch = CFG.per_device_train_batch_size * CFG.gradient_accumulation_steps
steps_per_epoch = math.ceil(len(train_dataset) / effective_batch)
total_steps = max(1, int(steps_per_epoch * CFG.num_epochs))
warmup_steps = max(1, int(total_steps * CFG.warmup_ratio))
print(
    f"effective batch {effective_batch} | steps/epoch {steps_per_epoch} | "
    f"total {total_steps} | warmup {warmup_steps}"
)

# The Trainer's eval pass runs the whole eval split at every eval point. On a
# 1,300-row split that is thousands of forward passes spent only to feed early
# stopping a single number. Cap it; section 10 still evaluates against the full
# split by generation, which is the measurement that actually matters.
trainer_eval_dataset = eval_dataset
if CFG.trainer_eval_max_rows and len(eval_dataset) > CFG.trainer_eval_max_rows:
    trainer_eval_dataset = eval_dataset.shuffle(seed=CFG.seed).select(
        range(CFG.trainer_eval_max_rows)
    )
    print(
        f"Trainer eval subset: {len(trainer_eval_dataset)} of {len(eval_dataset)} rows "
        "(early-stopping signal only)"
    )


class TimeBudgetCallback(TrainerCallback):
    """Stop training once the wall-clock budget is spent.

    The check runs on every step, not only at epoch end. With the full corpus a
    single epoch is a few hundred steps and several hours, so an epoch-boundary
    check would sail straight past the budget and never fire.

    Stopping mid-epoch is safe here because `load_best_model_at_end=True` reloads
    the best evaluated checkpoint, so what gets exported is a checkpoint that was
    measured -- at most `save_steps` behind the stopping point -- never a
    half-trained snapshot that no evaluation ever saw.
    """

    def __init__(self, budget_minutes: float) -> None:
        self.budget_seconds = budget_minutes * 60
        self.started = time.time()
        self.stopped_early = False
        self.stopped_at_step: int | None = None

    def _over_budget(self) -> bool:
        return bool(self.budget_seconds) and (time.time() - self.started) >= self.budget_seconds

    def _stop(self, state, control, where: str):  # noqa: ANN001
        elapsed = (time.time() - self.started) / 60
        print(
            f"\nTime budget reached: {elapsed:.1f} min elapsed at {where}. "
            "Stopping now; the best evaluated checkpoint will be loaded."
        )
        control.should_training_stop = True
        self.stopped_early = True
        self.stopped_at_step = state.global_step
        return control

    def on_step_end(self, args, state, control, **kwargs):  # noqa: ANN001, ANN003
        if not self.stopped_early and self._over_budget():
            return self._stop(state, control, f"step {state.global_step}")
        return control

    def on_epoch_end(self, args, state, control, **kwargs):  # noqa: ANN001, ANN003
        if not self.stopped_early and self._over_budget():
            return self._stop(state, control, f"epoch {state.epoch:.2f}")
        return control

    def on_log(self, args, state, control, logs=None, **kwargs):  # noqa: ANN001, ANN003
        if logs and "loss" in logs and self.budget_seconds:
            elapsed = (time.time() - self.started) / 60
            remaining = max(0.0, self.budget_seconds / 60 - elapsed)
            step = max(1, state.global_step)
            projected = step * (self.budget_seconds / 60) / max(elapsed, 1e-6)
            print(
                f"    [{elapsed:.0f}/{self.budget_seconds / 60:.0f} min, "
                f"{remaining:.0f} min left, step {step}, "
                f"projected ~{projected:.0f} steps in budget]"
            )
        return control

args_kwargs: dict[str, Any] = dict(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=CFG.num_epochs,
    per_device_train_batch_size=CFG.per_device_train_batch_size,
    per_device_eval_batch_size=CFG.per_device_eval_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    learning_rate=CFG.learning_rate,
    warmup_steps=warmup_steps,
    weight_decay=CFG.weight_decay,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    bf16=TORCH_BF16,
    fp16=not TORCH_BF16,
    logging_steps=CFG.logging_steps,
    save_strategy=CFG.checkpoint_strategy,
    save_total_limit=CFG.save_total_limit,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=CFG.max_grad_norm,
    group_by_length=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=CFG.seed,
)
# load_best_model_at_end requires the eval and save strategies to match, so both
# follow CFG.checkpoint_strategy. transformers renamed evaluation_strategy ->
# eval_strategy along the way; support both spellings.
_ta_params = inspect.signature(TrainingArguments.__init__).parameters
_eval_key = "eval_strategy" if "eval_strategy" in _ta_params else "evaluation_strategy"
args_kwargs[_eval_key] = CFG.checkpoint_strategy
if CFG.checkpoint_strategy == "steps":
    args_kwargs["save_steps"] = CFG.save_steps
    args_kwargs["eval_steps"] = CFG.eval_steps

# TrainingArguments gains and loses fields across transformers releases
# (group_by_length and evaluation_strategy are both casualties in recent ones).
# Rather than pin a version, keep only what the installed signature accepts and
# report the rest. Everything dropped here is a speed or logging convenience,
# never a correctness knob -- the optimizer, schedule, precision, masking, and
# checkpoint settings are all in the accepted set or the run would be wrong.
if not any(p.kind is p.VAR_KEYWORD for p in _ta_params.values()):
    _unsupported = sorted(key for key in args_kwargs if key not in _ta_params)
    for _key in _unsupported:
        args_kwargs.pop(_key)
    if _unsupported:
        print(f"transformers {transformers.__version__} does not accept: {_unsupported}")
        print("Dropped them and continued.")
        if "group_by_length" in _unsupported:
            print(
                "  group_by_length only batched similar-length rows for speed; "
                "without it training is slightly slower, not different."
            )

training_args = TrainingArguments(**args_kwargs)
callbacks: list[Any] = [
    EarlyStoppingCallback(early_stopping_patience=CFG.early_stopping_patience)
]
budget_callback = None
if CFG.target_train_minutes:
    budget_callback = TimeBudgetCallback(CFG.target_train_minutes)
    callbacks.append(budget_callback)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=trainer_eval_dataset,
    data_collator=AssistantOnlyCollator(tokenizer),
    callbacks=callbacks,
)

started = time.time()
train_result = trainer.train(resume_from_checkpoint=CFG.resume_from_checkpoint)
elapsed_minutes = (time.time() - started) / 60

# load_best_model_at_end=True means trainer.model is already the best checkpoint
# by eval_loss, so what gets saved here -- and therefore merged and exported to
# GGUF -- is the best epoch, not the last one.
best_checkpoint = trainer.state.best_model_checkpoint
best_metric = trainer.state.best_metric

trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
final_eval = trainer.evaluate()
trainer.save_metrics("train", train_result.metrics)
trainer.save_metrics("eval", final_eval)
trainer.save_state()

print()
print(f"Training finished in {elapsed_minutes:.1f} minutes")
print(f"epochs completed: {trainer.state.epoch:.2f} of {CFG.num_epochs:g}")
print(f"optimizer steps:  {trainer.state.global_step}")
print("train_loss:", train_result.metrics.get("train_loss"))
print("eval_loss: ", final_eval.get("eval_loss"))
if budget_callback and budget_callback.stopped_early:
    print(f"stopped by the {CFG.target_train_minutes} min budget, not by epoch count")
if best_checkpoint:
    print()
    print("Best epoch selected by eval_loss:")
    print("  checkpoint:", best_checkpoint)
    print("  eval_loss: ", best_metric)
    print("  This checkpoint is what gets merged and exported to GGUF.")
else:
    print()
    print("NOTE: no best checkpoint recorded; the final weights will be exported.")
    print("That happens when only one evaluation ran. Numbers remain valid.")

# Per-epoch history, so the write-up can show which epoch won and by how much.
epoch_rows = [
    (h.get("epoch"), h.get("eval_loss"))
    for h in trainer.state.log_history
    if "eval_loss" in h
]
if epoch_rows:
    print()
    print("  epoch    eval_loss")
    for epoch_value, loss_value in epoch_rows:
        marker = "  <-- best" if loss_value == best_metric else ""
        print(f"  {epoch_value:>6.2f}   {loss_value:.6f}{marker}")

print()
print("Adapter saved to:", ADAPTER_DIR)

transformers: 5.14.1
effective batch 16 | steps/epoch 266 | total 798 | warmup 39
Trainer eval subset: 256 of 642 rows (early-stopping signal only)
transformers 5.14.1 does not accept: ['group_by_length']
Dropped them and continued.
  group_by_length only batched similar-length rows for speed; without it training is slightly slower, not different.


Step,Training Loss,Validation Loss
40,0.014996,0.006662
80,0.003337,0.002701
120,0.003099,0.001920
160,0.002494,0.001370
200,0.002024,0.001119
240,0.001640,0.001316
280,0.002431,0.001763
320,0.001487,0.000904


    [2/240 min, 238 min left, step 5, projected ~717 steps in budget]
    [3/240 min, 237 min left, step 10, projected ~730 steps in budget]
    [5/240 min, 235 min left, step 15, projected ~755 steps in budget]
    [6/240 min, 234 min left, step 20, projected ~768 steps in budget]
    [8/240 min, 232 min left, step 25, projected ~771 steps in budget]
    [9/240 min, 231 min left, step 30, projected ~775 steps in budget]
    [11/240 min, 229 min left, step 35, projected ~773 steps in budget]
    [12/240 min, 228 min left, step 40, projected ~771 steps in budget]
    [16/240 min, 224 min left, step 45, projected ~688 steps in budget]
    [17/240 min, 223 min left, step 50, projected ~694 steps in budget]
    [19/240 min, 221 min left, step 55, projected ~702 steps in budget]
    [20/240 min, 220 min left, step 60, projected ~707 steps in budget]
    [22/240 min, 218 min left, step 65, projected ~709 steps in budget]
    [24/240 min, 216 min left, step 70, projected ~711 steps in budget]

Step,Training Loss,Validation Loss
40,0.014996,0.006662
80,0.003337,0.002701
120,0.003099,0.001920
160,0.002494,0.001370
200,0.002024,0.001119
240,0.001640,0.001316
280,0.002431,0.001763
320,0.001487,0.000904
360,0.003633,0.000605
400,0.001625,0.000722


    [123/240 min, 117 min left, step 350, projected ~685 steps in budget]
    [124/240 min, 116 min left, step 355, projected ~686 steps in budget]
    [126/240 min, 114 min left, step 360, projected ~688 steps in budget]
    [129/240 min, 111 min left, step 365, projected ~680 steps in budget]
    [130/240 min, 110 min left, step 370, projected ~681 steps in budget]
    [132/240 min, 108 min left, step 375, projected ~682 steps in budget]
    [133/240 min, 107 min left, step 380, projected ~683 steps in budget]
    [135/240 min, 105 min left, step 385, projected ~685 steps in budget]
    [137/240 min, 103 min left, step 390, projected ~686 steps in budget]
    [138/240 min, 102 min left, step 395, projected ~686 steps in budget]
    [140/240 min, 100 min left, step 400, projected ~687 steps in budget]
    [143/240 min, 97 min left, step 405, projected ~680 steps in budget]
    [145/240 min, 95 min left, step 410, projected ~681 steps in budget]
    [146/240 min, 94 min left, step 415,

Training Loss,Validation Loss,Step
0.000036,0.000106,681



Training finished in 240.3 minutes
epochs completed: 2.56 of 3
optimizer steps:  681
train_loss: 0.014868109009612574
eval_loss:  0.0001055971224559471
stopped by the 240 min budget, not by epoch count

Best epoch selected by eval_loss:
  checkpoint: /content/drive/MyDrive/bot-training/training_runs/fastAPI_Model-runtime-task/checkpoints/checkpoint-680
  eval_loss:  0.0001055971224559471
  This checkpoint is what gets merged and exported to GGUF.

  epoch    eval_loss
    0.15   0.006662
    0.30   0.002701
    0.45   0.001920
    0.60   0.001370
    0.75   0.001119
    0.90   0.001316
    1.05   0.001763
    1.20   0.000904
    1.35   0.000605
    1.50   0.000722
    1.66   0.000392
    1.81   0.000426
    1.96   0.000245
    2.11   0.000220
    2.26   0.000232
    2.41   0.000110
    2.56   0.000106  <-- best
    2.56   0.000106  <-- best

Adapter saved to: /content/drive/MyDrive/bot-training/training_runs/fastAPI_Model-runtime-task/lora_adapter


In [21]:
# Loss curve, written to disk so it can go straight into the thesis figures.
history = trainer.state.log_history
train_points = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_points = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

(REPORT_DIR / "loss_history.json").write_text(
    json.dumps(
        {"train": train_points, "eval": eval_points, "elapsed_minutes": elapsed_minutes},
        indent=2,
    ),
    encoding="utf-8",
)

try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    figure, axis = plt.subplots(figsize=(8, 4.5), dpi=150)
    if train_points:
        axis.plot(*zip(*train_points), label="train loss", linewidth=1.2)
    if eval_points:
        axis.plot(*zip(*eval_points), label="eval loss", marker="o", linewidth=1.4)
    axis.set_xlabel("step")
    axis.set_ylabel("loss")
    axis.set_title(f"{CFG.model_name} LoRA - runtime-task.v1")
    axis.grid(alpha=0.3)
    axis.legend()
    figure.tight_layout()
    curve_path = REPORT_DIR / "loss_curve.png"
    figure.savefig(curve_path)
    plt.close(figure)
    print("Loss curve:", curve_path)
except ImportError:
    print("matplotlib not installed; loss_history.json written without a plot.")

Loss curve: /content/drive/MyDrive/bot-training/training_runs/fastAPI_Model-runtime-task/reports/loss_curve.png


## 10. Per-stage generation evaluation


In [22]:
model.eval()
model.config.use_cache = True

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
eos_ids = [tokenizer.eos_token_id]
if isinstance(im_end_id, int) and im_end_id >= 0 and im_end_id not in eos_ids:
    eos_ids.append(im_end_id)


def generate_answer(messages: list[dict[str, str]], *, max_new_tokens: int | None = None) -> str:
    prompt_messages = [m for m in messages if m["role"] != "assistant"]
    prompt_text = render_chat(prompt_messages, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(
        next(model.parameters()).device
    )
    input_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens or CFG.max_new_tokens,
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.02,
        )
    return tokenizer.decode(output[0][input_length:], skip_special_tokens=True).strip()


def parse_prediction(task: str, text: str) -> tuple[Any, str | None, str | None]:
    """Return (payload, parse_error, contract_error)."""
    if task in TEXT_TASKS:
        if not FILE_WRAPPER_RE.search(text or ""):
            return None, "missing_file_wrapper", None
        payload: Any = text
    else:
        try:
            payload = json.loads(extract_json_object(text))
        except Exception as exc:  # noqa: BLE001
            return None, f"json:{exc}", None
    try:
        validate_task_payload(task, payload, "prediction")
    except Exception as exc:  # noqa: BLE001
        return payload, None, str(exc)
    return payload, None, None


# Sample the eval set proportionally so every present stage is represented.
by_task_eval: dict[str, list[dict[str, Any]]] = collections.defaultdict(list)
for record in eval_records:
    by_task_eval[record["task"]].append(record)

sampled_eval: list[dict[str, Any]] = []
if CFG.eval_sample_limit is None:
    sampled_eval = list(eval_records)
else:
    per_task = max(1, CFG.eval_sample_limit // max(1, len(by_task_eval)))
    for task, rows in by_task_eval.items():
        sampled_eval.extend(random.Random(CFG.seed).sample(rows, min(per_task, len(rows))))

print(f"Evaluating {len(sampled_eval)} records across {len(by_task_eval)} stages...")

predictions_path = REPORT_DIR / "eval_predictions.jsonl"
per_task_metrics: dict[str, collections.Counter] = collections.defaultdict(collections.Counter)
per_task_f1: dict[str, dict[str, list[float]]] = collections.defaultdict(
    lambda: collections.defaultdict(list)
)
failures: list[dict[str, Any]] = []

with predictions_path.open("w", encoding="utf-8") as handle:
    for position, record in enumerate(sampled_eval, start=1):
        task = record["task"]
        reference_text = record["messages"][-1]["content"]
        reference = (
            reference_text if task in TEXT_TASKS else json.loads(extract_json_object(reference_text))
        )
        prediction_text = generate_answer(record["messages"])
        prediction, parse_error, contract_error = parse_prediction(task, prediction_text)

        parsed_ok = parse_error is None
        contract_ok = parsed_ok and contract_error is None
        exact = contract_ok and (
            prediction_text.strip() == reference_text.strip()
            if task in TEXT_TASKS
            else stable_hash(prediction) == stable_hash(reference)
        )

        metrics = per_task_metrics[task]
        metrics["total"] += 1
        metrics["parsed"] += int(parsed_ok)
        metrics["contract_ok"] += int(contract_ok)
        metrics["exact"] += int(exact)

        if parsed_ok:
            predicted_signature = structural_signature(task, prediction)
            reference_signature = structural_signature(task, reference)
            for component, gold in reference_signature.items():
                per_task_f1[task][component].append(
                    f1(predicted_signature.get(component, set()), gold)
                )

        handle.write(
            canonical_json(
                {
                    "task": task,
                    "source": record["source"],
                    "user": record["messages"][1]["content"],
                    "prediction_text": prediction_text,
                    "parse_error": parse_error,
                    "contract_error": contract_error,
                    "parsed": parsed_ok,
                    "contract_ok": contract_ok,
                    "exact_match": exact,
                }
            )
            + "\n"
        )
        if not contract_ok and len(failures) < 40:
            failures.append(
                {
                    "task": task,
                    "user": record["messages"][1]["content"][:300],
                    "prediction": prediction_text[:800],
                    "parse_error": parse_error,
                    "contract_error": contract_error,
                }
            )
        if position % 20 == 0:
            print(f"  {position}/{len(sampled_eval)}")

report = {
    "protocol": PROTOCOL["version"],
    "base_model": CFG.base_model_id,
    "memory_profile": CFG.memory_profile,
    "eval_loss": final_eval.get("eval_loss"),
    "train_minutes": round(elapsed_minutes, 2),
    "per_task": {},
}
for task, metrics in per_task_metrics.items():
    total = metrics["total"] or 1
    report["per_task"][task] = {
        "examples": metrics["total"],
        "parse_rate": round(metrics["parsed"] / total, 4),
        "contract_rate": round(metrics["contract_ok"] / total, 4),
        "exact_match_rate": round(metrics["exact"] / total, 4),
        "component_f1": {
            component: round(sum(values) / len(values), 4)
            for component, values in sorted(per_task_f1[task].items())
        },
    }

overall_total = sum(m["total"] for m in per_task_metrics.values()) or 1
report["overall"] = {
    "examples": overall_total,
    "parse_rate": round(sum(m["parsed"] for m in per_task_metrics.values()) / overall_total, 4),
    "contract_rate": round(
        sum(m["contract_ok"] for m in per_task_metrics.values()) / overall_total, 4
    ),
    "exact_match_rate": round(sum(m["exact"] for m in per_task_metrics.values()) / overall_total, 4),
}

(REPORT_DIR / "eval_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print()
print(json.dumps(report, indent=2, ensure_ascii=False))
print("Predictions:", predictions_path)

Evaluating 90 records across 3 stages...
  20/90
  40/90
  60/90
  80/90

{
  "protocol": "runtime-task.v1",
  "base_model": "Qwen/Qwen2.5-Coder-3B-Instruct",
  "memory_profile": "GPU_16GB",
  "eval_loss": 0.0001055971224559471,
  "train_minutes": 240.34,
  "per_task": {
    "file_plan": {
      "examples": 30,
      "parse_rate": 1.0,
      "contract_rate": 1.0,
      "exact_match_rate": 1.0,
      "component_f1": {
        "files": 1.0
      }
    },
    "schema_plan": {
      "examples": 30,
      "parse_rate": 1.0,
      "contract_rate": 1.0,
      "exact_match_rate": 0.9667,
      "component_f1": {
        "columns": 0.9989,
        "filters": 1.0,
        "relationships": 1.0,
        "tables": 1.0
      }
    },
    "api_contract": {
      "examples": 30,
      "parse_rate": 1.0,
      "contract_rate": 1.0,
      "exact_match_rate": 0.9667,
      "component_f1": {
        "auth": 1.0,
        "required_fields": 0.98,
        "required_filters": 1.0,
        "required_routes": 0.

### 10.1 Failure clustering


In [23]:
def classify_failure(row: dict[str, Any]) -> list[str]:
    tags: list[str] = []
    if row.get("parse_error"):
        error = str(row["parse_error"])
        tags.append("unparseable")
        if error.startswith("missing_file_wrapper"):
            tags.append("wrapper_missing")
        elif "no complete JSON object" in error:
            tags.append("truncated_output")
        else:
            tags.append("malformed_json")
        return tags
    contract_error = str(row.get("contract_error") or "")
    if "missing required keys" in contract_error:
        tags.append("missing_required_keys")
    if "unsupported op" in contract_error:
        tags.append("illegal_patch_op")
    if "omits required files" in contract_error:
        tags.append("file_plan_incomplete")
    if "unknown keys" in contract_error:
        tags.append("artifact_contract_key_drift")
    if "ddl_sql" in contract_error:
        tags.append("ddl_schema_mismatch")
    if "routes is empty" in contract_error:
        tags.append("no_routes")
    return tags or ["structural_mismatch"]


clusters: collections.Counter = collections.Counter()
cluster_by_task: dict[str, collections.Counter] = collections.defaultdict(collections.Counter)
examples_by_cluster: dict[str, list[dict[str, Any]]] = collections.defaultdict(list)

with predictions_path.open("r", encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        if row["contract_ok"] and row["exact_match"]:
            continue
        for tag in classify_failure(row):
            clusters[tag] += 1
            cluster_by_task[row["task"]][tag] += 1
            if len(examples_by_cluster[tag]) < 3:
                examples_by_cluster[tag].append(
                    {"task": row["task"], "user": row["user"][:240], "prediction": row["prediction_text"][:500]}
                )

failure_report = {
    "clusters": clusters.most_common(),
    "clusters_by_task": {task: counter.most_common() for task, counter in cluster_by_task.items()},
    "examples": dict(examples_by_cluster),
}
(REPORT_DIR / "failure_clusters.json").write_text(
    json.dumps(failure_report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(failure_report["clusters"], indent=2, ensure_ascii=False))
print()
for task, counter in cluster_by_task.items():
    print(f"{task}: {counter.most_common(4)}")

[
  [
    "structural_mismatch",
    2
  ]
]

schema_plan: [('structural_mismatch', 1)]
api_contract: [('structural_mismatch', 1)]


## 11. Merge the adapter into full weights


In [28]:
import gc  # noqa: E402
import inspect  # noqa: E402

from peft import PeftModel  # noqa: E402

gc.collect()
torch.cuda.empty_cache()
print("GPU released. Reserved (GiB):", round(torch.cuda.memory_reserved() / 1024**3, 2))

if CFG.merge_adapter:
    merge_device = "cpu" if CFG.merge_on_cpu else "auto"
    print(f"Loading the base model for merge on {merge_device}...")

    # transformers 5 renamed from_pretrained's `torch_dtype` to `dtype`. This is
    # not cosmetic here: if the keyword is silently ignored the model loads in
    # fp32 at ~12.5 GB, which will not merge on a 15 GB T4 alongside the ~2.5 GB
    # the training loop leaves reserved. Pick the accepted spelling, then verify
    # the parameters really came back as fp16 instead of trusting the call.
    _fp_params = inspect.signature(AutoModelForCausalLM.from_pretrained).parameters
    _dtype_key = "dtype" if "dtype" in _fp_params else "torch_dtype"
    load_kwargs: dict[str, Any] = {
        _dtype_key: torch.float16,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }
    if merge_device != "cpu":
        load_kwargs["device_map"] = merge_device

    try:
        base_model = AutoModelForCausalLM.from_pretrained(CFG.base_model_id, **load_kwargs)
    except TypeError as exc:
        print(f"{_dtype_key} rejected ({exc}); retrying with the other spelling.")
        load_kwargs.pop(_dtype_key)
        other = "torch_dtype" if _dtype_key == "dtype" else "dtype"
        load_kwargs[other] = torch.float16
        base_model = AutoModelForCausalLM.from_pretrained(CFG.base_model_id, **load_kwargs)

    loaded_dtype = next(base_model.parameters()).dtype
    print(f"Loaded via {_dtype_key}= | parameter dtype: {loaded_dtype}")
    if loaded_dtype != torch.float16:
        print(f"WARNING: expected float16 but got {loaded_dtype}; casting down.")
        base_model = base_model.half()

    merged = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
    merged = merged.merge_and_unload()
    merged.save_pretrained(str(MERGED_DIR), safe_serialization=True, max_shard_size="2GB")
    tokenizer.save_pretrained(str(MERGED_DIR))

    del merged
    del base_model
    gc.collect()
    torch.cuda.empty_cache()
    print("Merged weights:", MERGED_DIR)
    print("Files:", sorted(p.name for p in MERGED_DIR.iterdir()))
else:
    print("CFG.merge_adapter is False (rehearsal). Skipping the merge.")
    print("The adapter is saved to Drive and can be merged in a later session:")
    print("   ", ADAPTER_DIR)

GPU released. Reserved (GiB): 7.01
Loading the base model for merge on auto...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded via torch_dtype= | parameter dtype: torch.float16


Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Merged weights: /content/merged_hf
Files: ['chat_template.jinja', 'config.json', 'generation_config.json', 'model-00001-of-00004.safetensors', 'model-00002-of-00004.safetensors', 'model-00003-of-00004.safetensors', 'model-00004-of-00004.safetensors', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json']


## 12. Convert to GGUF and quantize


In [1]:
# --- 12.0 Resume after a runtime restart -----------------------------------
# A restart empties /content: the merged fp16 weights, the llama.cpp checkout,
# and every Python variable sections 12-14 read are gone. What survives is the
# Drive run folder, and it holds enough to rebuild all of it without touching
# the GPU for anything but the merge:
#
#   lora_adapter/            the trained weights and the exact tokenizer
#   data/runtime_task_*.jsonl the built train/eval splits
#   reports/eval_report.json  the section-10 metrics, incl. eval_loss
#   reports/loss_history.json elapsed_minutes and the loss curve points
#   checkpoints/*/trainer_state.json  the epochs/steps actually run
#
# Re-running sections 1-11 would also work, but section 9 is the four hours
# this cell exists to skip. It is a no-op when the session state is intact, so
# it is safe to leave in place and run on a normal, uninterrupted run.
#
# A CPU-only runtime is enough for everything from here on if the merge is done
# on CPU; only the merge itself benefits from the GPU.

RESUME_RUN_SLUG = "fastAPI_Model-runtime-task"  # a rehearsal run ends in "-rehearsal"

# Short-circuits before touching MERGED_DIR, which does not exist after a restart.
_state_ok = all(n in globals() for n in ("CFG", "report", "MERGED_DIR", "train_records"))
_merged_ok = _state_ok and MERGED_DIR.exists() and any(MERGED_DIR.glob("*.safetensors"))

if _merged_ok:
    RESUMED = False
    print("Session state is intact and the merged weights are on disk; nothing to resume.")
else:
    RESUMED = True
    import gc  # noqa: E402
    import hashlib  # noqa: F401,E402  (section 13 needs it)
    import inspect  # noqa: E402
    import json  # noqa: E402
    import os  # noqa: E402
    import shutil  # noqa: E402
    import subprocess  # noqa: E402
    import sys  # noqa: E402
    import types  # noqa: E402
    from pathlib import Path  # noqa: E402
    from typing import Any  # noqa: F401,E402

    import torch  # noqa: E402

    # --- paths ------------------------------------------------------------
    DRIVE_ROOT = Path("/content/drive/MyDrive/bot-training")
    if not DRIVE_ROOT.exists():
        raise FileNotFoundError(
            f"{DRIVE_ROOT} not found. Mount Drive first:\n"
            "    from google.colab import drive; drive.mount('/content/drive')"
        )

    RUN_ROOT = DRIVE_ROOT / "training_runs" / RESUME_RUN_SLUG
    if not RUN_ROOT.exists():
        available = sorted(p.name for p in (DRIVE_ROOT / "training_runs").glob("*"))
        raise FileNotFoundError(f"{RUN_ROOT} not found. Runs on Drive: {available}")

    BUILT_DATA_DIR = RUN_ROOT / "data"
    CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
    ADAPTER_DIR = RUN_ROOT / "lora_adapter"
    REPORT_DIR = RUN_ROOT / "reports"
    EXPORT_DIR = RUN_ROOT / "export"
    TRAIN_JSONL = BUILT_DATA_DIR / "runtime_task_train.jsonl"
    EVAL_JSONL = BUILT_DATA_DIR / "runtime_task_eval.jsonl"
    MERGED_DIR = Path("/content/merged_hf")
    LLAMA_CPP_DIR = Path("/content/llama.cpp")
    for _path in (EXPORT_DIR, MERGED_DIR):
        _path.mkdir(parents=True, exist_ok=True)

    if not (ADAPTER_DIR / "adapter_config.json").exists():
        raise FileNotFoundError(
            f"No LoRA adapter at {ADAPTER_DIR}. Without it the run cannot be resumed "
            "and section 9 has to run again."
        )
    REHEARSAL = RESUME_RUN_SLUG.endswith("-rehearsal")

    def _read_json(path: Path, default=None):
        try:
            return json.loads(path.read_text(encoding="utf-8-sig"))
        except (OSError, ValueError):
            return default

    def _read_jsonl(path: Path) -> list[dict]:
        if not path.exists():
            raise FileNotFoundError(f"{path} is missing; re-run sections 5-7 to rebuild the splits.")
        with path.open("r", encoding="utf-8") as handle:
            return [json.loads(line) for line in handle if line.strip()]

    # --- protocol ---------------------------------------------------------
    PROTOCOL_JSON = next(
        (p for p in (DRIVE_ROOT / "training_protocol.json",
                     DRIVE_ROOT / "training_data" / "training_protocol.json") if p.exists()),
        None,
    ) or next(DRIVE_ROOT.rglob("training_protocol.json"), None)
    if PROTOCOL_JSON is None:
        raise FileNotFoundError(f"training_protocol.json not found under {DRIVE_ROOT}")
    PROTOCOL = json.loads(PROTOCOL_JSON.read_text(encoding="utf-8-sig"))
    ALL_TASKS = tuple(PROTOCOL["tasks"].keys())

    # --- corpus and metrics, straight off Drive ---------------------------
    train_records = _read_jsonl(TRAIN_JSONL)
    eval_records = _read_jsonl(EVAL_JSONL)

    report = _read_json(REPORT_DIR / "eval_report.json")
    if report is None:
        raise FileNotFoundError(
            f"{REPORT_DIR / 'eval_report.json'} is missing; re-run section 10 (generation eval) "
            "before the export, or the manifest would carry no measured numbers."
        )
    loss_history = _read_json(REPORT_DIR / "loss_history.json", {}) or {}

    # eval_loss and wall clock come from the run that produced the adapter, never
    # from this session -- the manifest has to describe the training, not the export.
    _eval_points = loss_history.get("eval", [])
    final_eval = {
        "eval_loss": report.get("eval_loss")
        if report.get("eval_loss") is not None
        else (_eval_points[-1][1] if _eval_points else None)
    }
    elapsed_minutes = float(
        loss_history.get("elapsed_minutes") or report.get("train_minutes") or 0.0
    )

    # --- config stand-in --------------------------------------------------
    # Only the fields sections 12-14 read. Values come from what the run itself
    # recorded where that exists (adapter_config.json is authoritative for the
    # LoRA shape, eval_report.json for the base model and memory profile) and
    # from section 3's full-run settings otherwise.
    _adapter_cfg = _read_json(ADAPTER_DIR / "adapter_config.json", {}) or {}
    _states = sorted(
        CHECKPOINT_DIR.glob("checkpoint-*/trainer_state.json"),
        key=lambda p: int(p.parent.name.split("-")[-1]),
    )
    _trainer_state = _read_json(_states[-1], {}) if _states else {}

    CFG = types.SimpleNamespace(
        model_name="fastAPI_Model",
        base_model_id=_adapter_cfg.get("base_model_name_or_path")
        or report.get("base_model", "Qwen/Qwen2.5-Coder-3B-Instruct"),
        memory_profile=report.get("memory_profile", "GPU_16GB"),
        quantization="Q4_K_M",
        lora_r=_adapter_cfg.get("r", 32),
        lora_alpha=_adapter_cfg.get("lora_alpha", 64),
        lora_dropout=_adapter_cfg.get("lora_dropout", 0.05),
        num_epochs=float(_trainer_state.get("num_train_epochs") or 3.0),
        max_seq_length=1024,  # section 3 caps the full run at 1024
        learning_rate=1.5e-4,
        seed=42,
        merge_adapter=True,
        merge_on_cpu=not torch.cuda.is_available(),
        export_gguf=True,
        keep_f16_gguf=True,
        copy_merged_to_drive=False,
        copy_f16_gguf_to_drive=False,
    )

    # --- hardware of THIS session ----------------------------------------
    if torch.cuda.is_available():
        _props = torch.cuda.get_device_properties(0)
        GPU_NAME = torch.cuda.get_device_name(0)
        GPU_VRAM_MB = int(_props.total_memory / 1024**2)
        GPU_COMPUTE = f"{_props.major}.{_props.minor}"
        COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        GPU_NAME, GPU_VRAM_MB, GPU_COMPUTE = "cpu", 0, "n/a"
        COMPUTE_DTYPE = torch.float32

    print(f"Resuming {RESUME_RUN_SLUG} from Drive")
    print(f"  adapter        {ADAPTER_DIR}")
    print(f"  corpus         {len(train_records)} train / {len(eval_records)} eval rows")
    print(f"  eval_loss      {final_eval['eval_loss']}")
    print(f"  contract_rate  {report['overall']['contract_rate']}")
    print(f"  train_minutes  {elapsed_minutes:.2f}")
    print(f"  this session   {GPU_NAME} ({GPU_VRAM_MB} MB)")

    # --- re-merge the adapter --------------------------------------------
    if MERGED_DIR.exists() and any(MERGED_DIR.glob("*.safetensors")):
        print("\nMerged fp16 weights already on local disk; skipping the merge.")
    else:
        try:
            from peft import PeftModel  # noqa: E402
            from transformers import AutoModelForCausalLM, AutoTokenizer  # noqa: E402
        except ImportError:
            print("\nInstalling peft/transformers into the fresh runtime...")
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q",
                 "transformers>=4.44.0", "peft>=0.12.0", "accelerate>=0.34.0"],
                check=False,
            )
            from peft import PeftModel  # noqa: E402
            from transformers import AutoModelForCausalLM, AutoTokenizer  # noqa: E402

        # Colab ships torchao 0.10 while current peft refuses to run beside
        # anything below 0.16, and it raises that from deep inside
        # PeftModel.from_pretrained -- after the 6 GB base model has already been
        # downloaded and loaded. Nothing here uses torchao; it matters only for
        # torchao quantization, and upgrading it would pull a different torch
        # build into a working runtime. So ask peft's own check first, and if it
        # objects, remove the package rather than the torch underneath it.
        try:
            from peft.import_utils import is_torchao_available  # noqa: E402
        except Exception:  # noqa: BLE001 - older peft has no such helper
            is_torchao_available = None
        if is_torchao_available is not None:
            try:
                is_torchao_available()
            except ImportError as exc:
                print(f"peft rejects the installed torchao ({exc}); removing it.")
                subprocess.run(
                    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                    check=False,
                )

        merge_device = "cpu" if CFG.merge_on_cpu else "auto"
        print(f"\nRe-merging the adapter on {merge_device} (~5 min, no retraining)...")
        if merge_device == "cpu":
            print("  CPU merge: a 3B fp16 merge needs ~7 GB of RAM. On a standard "
                  "CPU runtime this is close to the limit; a GPU runtime is safer.")

        # transformers 5 renamed from_pretrained's `torch_dtype` to `dtype`; passing
        # the wrong one is silently ignored and the model loads fp32 at ~12.5 GB.
        _fp_params = inspect.signature(AutoModelForCausalLM.from_pretrained).parameters
        _dtype_key = "dtype" if "dtype" in _fp_params else "torch_dtype"
        load_kwargs: dict = {
            _dtype_key: torch.float16,
            "trust_remote_code": True,
            "low_cpu_mem_usage": True,
        }
        if merge_device != "cpu":
            load_kwargs["device_map"] = merge_device

        base_model = AutoModelForCausalLM.from_pretrained(CFG.base_model_id, **load_kwargs)
        if next(base_model.parameters()).dtype != torch.float16:
            base_model = base_model.half()

        merged = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
        merged = merged.merge_and_unload()
        merged.save_pretrained(str(MERGED_DIR), safe_serialization=True, max_shard_size="2GB")
        # The tokenizer saved beside the adapter is the one training used, chat
        # template and added tokens included; the Hub copy is not guaranteed to match.
        AutoTokenizer.from_pretrained(str(ADAPTER_DIR)).save_pretrained(str(MERGED_DIR))

        del merged, base_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Merged weights:", MERGED_DIR)
        print("Files:", sorted(p.name for p in MERGED_DIR.iterdir()))

    print("\nState rebuilt. The next cell converts to GGUF; sections 13 and 14 then run as usual.")

Resuming fastAPI_Model-runtime-task from Drive
  adapter        /content/drive/MyDrive/bot-training/training_runs/fastAPI_Model-runtime-task/lora_adapter
  corpus         4247 train / 642 eval rows
  eval_loss      0.0001055971224559471
  contract_rate  1.0
  train_minutes  240.34
  this session   cpu (0 MB)

Re-merging the adapter on cpu (~5 min, no retraining)...
  CPU merge: a 3B fp16 merge needs ~7 GB of RAM. On a standard CPU runtime this is close to the limit; a GPU runtime is safer.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Merged weights: /content/merged_hf
Files: ['chat_template.jinja', 'config.json', 'generation_config.json', 'model-00001-of-00004.safetensors', 'model-00002-of-00004.safetensors', 'model-00003-of-00004.safetensors', 'model-00004-of-00004.safetensors', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json']

State rebuilt. The next cell converts to GGUF; sections 13 and 14 then run as usual.


In [2]:
GGUF_F16_PATH = Path("/content") / f"{CFG.model_name}.f16.gguf"
GGUF_QUANT_PATH = Path("/content") / f"{CFG.model_name}.{CFG.quantization}.gguf"
export_state: dict[str, Any] = {"route": None, "f16": None, "quantized": None}
if globals().get("RESUMED"):
    # So the run summary does not read as one uninterrupted session.
    export_state["resumed_from_drive"] = True


def build_llama_tools() -> Path:
    """Clone llama.cpp and build only the quantizer target."""
    if not (LLAMA_CPP_DIR / "convert_hf_to_gguf.py").exists():
        print("Cloning llama.cpp...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(LLAMA_CPP_DIR)],
            check=True,
        )
    # llama.cpp's requirements file pins numpy/torch/transformers to versions that
    # fight with what Colab already has; installing it either fails outright or
    # downgrades numpy underneath the running kernel, after which the converter
    # dies on import. Install only what the converter actually needs and is not
    # already present, and never let a resolver failure abort the export.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "protobuf"],
        check=False,
    )
    return LLAMA_CPP_DIR


def convert_env() -> dict[str, str]:
    """The converter imports the repo's vendored `gguf` package, not a pip-installed one."""
    env = dict(os.environ)
    gguf_py = LLAMA_CPP_DIR / "gguf-py"
    if gguf_py.exists():
        env["PYTHONPATH"] = os.pathsep.join(
            part for part in (str(gguf_py), env.get("PYTHONPATH", "")) if part
        )
    return env


def find_quantizer() -> Path | None:
    found = shutil.which("llama-quantize")
    if found:
        return Path(found)
    for candidate in LLAMA_CPP_DIR.rglob("llama-quantize"):
        if candidate.is_file() and os.access(candidate, os.X_OK):
            return candidate
    return None


def compile_quantizer() -> Path | None:
    """CPU-only build of the quantize target; no CUDA needed to quantize."""
    print("Building llama-quantize (a few minutes)...")
    try:
        subprocess.run(
            ["cmake", "-B", str(LLAMA_CPP_DIR / "build"), "-S", str(LLAMA_CPP_DIR),
             "-DLLAMA_CURL=OFF", "-DGGML_NATIVE=ON", "-DCMAKE_BUILD_TYPE=Release"],
            check=True, capture_output=True, text=True,
        )
        subprocess.run(
            ["cmake", "--build", str(LLAMA_CPP_DIR / "build"), "--target", "llama-quantize",
             "-j", str(os.cpu_count() or 2)],
            check=True, capture_output=True, text=True,
        )
    except subprocess.CalledProcessError as exc:
        print("Build failed:")
        print((exc.stderr or "")[-2500:])
        return None
    return find_quantizer()


if CFG.export_gguf and CFG.merge_adapter:
    # The three artifacts together need ~14.5 GB: 6.2 merged + 6.2 f16 + 1.9 Q4.
    # Check first, because running out of disk halfway through the GGUF write
    # leaves a truncated file that still looks plausible on disk.
    free_gb = shutil.disk_usage("/content").free / 1e9
    needed_gb = 9.0 if CFG.keep_f16_gguf else 8.5
    print(f"Local disk free: {free_gb:.1f} GB | needed for this step: ~{needed_gb:.1f} GB")
    if free_gb < needed_gb:
        raise RuntimeError(
            f"Only {free_gb:.1f} GB free on /content but ~{needed_gb:.1f} GB is needed. "
            "Delete the merged weights or set CFG.keep_f16_gguf = False."
        )

    llama_cpp = build_llama_tools()
    convert_script = next(llama_cpp.rglob("convert_hf_to_gguf.py"), None)
    if convert_script is None:
        raise FileNotFoundError("convert_hf_to_gguf.py not found in the llama.cpp checkout")

    print("Converting merged weights to f16 GGUF...")
    # capture_output plus an explicit check: an uncaptured subprocess writes to the
    # kernel's stderr, which Colab never renders, so a failure surfaces only as
    # "returned non-zero exit status 1" with the actual traceback lost.
    convert = subprocess.run(
        [sys.executable, str(convert_script), str(MERGED_DIR),
         "--outfile", str(GGUF_F16_PATH), "--outtype", "f16"],
        env=convert_env(), capture_output=True, text=True,
    )
    if convert.returncode != 0:
        print((convert.stdout or "")[-2000:])
        print((convert.stderr or "")[-5000:])
        # A half-written GGUF still looks plausible on disk; delete it.
        GGUF_F16_PATH.unlink(missing_ok=True)
        raise RuntimeError("convert_hf_to_gguf.py failed - see the converter output above.")
    print((convert.stdout or "")[-1500:])
    export_state["f16"] = str(GGUF_F16_PATH)
    print("f16 GGUF:", GGUF_F16_PATH, f"({GGUF_F16_PATH.stat().st_size / 1e9:.2f} GB)")

    quantizer = find_quantizer() or compile_quantizer()
    if quantizer is None:
        raise RuntimeError(
            "Could not obtain llama-quantize. Re-run this cell, or quantize locally "
            "from the f16 GGUF copied to Drive."
        )
    print("Quantizing with", quantizer)
    quant = subprocess.run(
        [str(quantizer), str(GGUF_F16_PATH), str(GGUF_QUANT_PATH), CFG.quantization],
        capture_output=True, text=True,
    )
    if quant.returncode != 0:
        print((quant.stdout or "")[-2000:])
        print((quant.stderr or "")[-5000:])
        GGUF_QUANT_PATH.unlink(missing_ok=True)
        raise RuntimeError("llama-quantize failed - see the output above.")
    export_state["route"] = "llama-quantize"
    export_state["quantized"] = str(GGUF_QUANT_PATH)
    size_gb = GGUF_QUANT_PATH.stat().st_size / 1e9
    print(f"Quantized GGUF: {GGUF_QUANT_PATH} ({size_gb:.2f} GB)")

    if CFG.keep_f16_gguf:
        print("Keeping the f16 GGUF (CFG.keep_f16_gguf is True).")
    else:
        GGUF_F16_PATH.unlink(missing_ok=True)
        export_state["f16"] = None
        print("Removed the f16 intermediate to free disk.")

    export_state["merged_hf"] = str(MERGED_DIR)
    print()
    print("All three artifacts now exist on the local disk:")
    for label, path in (
        ("merged fp16 weights", MERGED_DIR),
        ("GGUF f16", GGUF_F16_PATH),
        (f"GGUF {CFG.quantization}", GGUF_QUANT_PATH),
    ):
        if path.exists():
            size = (
                sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
                if path.is_dir()
                else path.stat().st_size
            )
            print(f"  {label:<22} {size / 1e9:>6.2f} GB   {path}")
else:
    print("GGUF export skipped.")
    print("  export_gguf:", CFG.export_gguf, "| merge_adapter:", CFG.merge_adapter)

Local disk free: 81.0 GB | needed for this step: ~9.0 GB
Cloning llama.cpp...
Converting merged weights to f16 GGUF...

f16 GGUF: /content/fastAPI_Model.f16.gguf (6.18 GB)
Building llama-quantize (a few minutes)...
Quantizing with /content/llama.cpp/build/bin/llama-quantize
Quantized GGUF: /content/fastAPI_Model.Q4_K_M.gguf (1.93 GB)
Keeping the f16 GGUF (CFG.keep_f16_gguf is True).

All three artifacts now exist on the local disk:
  merged fp16 weights      6.18 GB   /content/merged_hf
  GGUF f16                 6.18 GB   /content/fastAPI_Model.f16.gguf
  GGUF Q4_K_M              1.93 GB   /content/fastAPI_Model.Q4_K_M.gguf


## 13. Build the export bundle on Drive


In [6]:
RUN_SLUG = "fastAPI_Model-runtime-task"

In [7]:
MODELFILE_TEXT = """FROM ./{artifact}

PARAMETER num_ctx 8192
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER stop <|im_end|>
PARAMETER stop <|endoftext|>

TEMPLATE \"\"\"{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{- end }}}}{{{{- range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{- end }}}}{{{{- if .Prompt }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
{{{{- end }}}}<|im_start|>assistant
\"\"\"

SYSTEM \"\"\"
You are the local FastAPI generation model for this project.

Follow the system instruction for the current pipeline stage exactly. A stage may
request schema JSON, an API contract, a file plan, complete file content, an edit
patch, or a focused repair. Return only the requested format. Never wrap JSON in
markdown and never add explanations around machine-readable output.

Preserve existing behavior unless the request explicitly changes it. Do not
invent unrelated resources or overwrite files outside the supplied scope. Code
must be complete, typed, maintainable, and runnable with correct imports. Prefer
small coherent changes and repair only the reported validation failures.
\"\"\"
"""


def sha256_file(path: Path, *, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


if export_state.get("quantized"):
    artifact_name = f"{CFG.model_name}.{CFG.quantization}.gguf"
    drive_artifact = EXPORT_DIR / artifact_name

    print(f"Copying {artifact_name} to Drive (about 2 GB, this takes a few minutes)...")
    shutil.copy2(export_state["quantized"], drive_artifact)

    print("Computing SHA-256 on the Drive copy...")
    artifact_hash = sha256_file(drive_artifact)
    artifact_size = drive_artifact.stat().st_size

    (EXPORT_DIR / "Modelfile.fastAPI_Model").write_text(
        MODELFILE_TEXT.format(artifact=f"models/{CFG.model_name}/{artifact_name}"),
        encoding="utf-8",
    )

    manifest = {
        "model_name": CFG.model_name,
        "artifact": f"models/{CFG.model_name}/{artifact_name}",
        "artifact_size_bytes": artifact_size,
        "sha256": artifact_hash,
        "base_family": "Qwen2.5 Coder",
        "parameter_class": "3B",
        "quantization": CFG.quantization,
        "ollama_context": 8192,
        "runtime_contract": PROTOCOL["runtime_contract"],
        "training_protocol": PROTOCOL["version"],
        "fine_tuned": True,
        "training_status": (
            f"LoRA fine-tuned on {len(train_records)} runtime-task.v1 records "
            f"({CFG.num_epochs} epochs, r={CFG.lora_r}, seq={CFG.max_seq_length}) on "
            f"{GPU_NAME} via Colab, merged to fp16 and re-quantized."
        ),
        "training_run": {
            "base_model": CFG.base_model_id,
            "platform": "google-colab",
            "gpu": GPU_NAME,
            "train_rows": len(train_records),
            "eval_rows": len(eval_records),
            "per_task_train": {
                task: sum(1 for r in train_records if r["task"] == task) for task in ALL_TASKS
            },
            "epochs": CFG.num_epochs,
            "lora_r": CFG.lora_r,
            "lora_alpha": CFG.lora_alpha,
            "max_seq_length": CFG.max_seq_length,
            "learning_rate": CFG.learning_rate,
            "eval_loss": final_eval.get("eval_loss"),
            "overall_contract_rate": report["overall"]["contract_rate"],
            "train_minutes": round(elapsed_minutes, 2),
            "memory_profile": CFG.memory_profile,
        },
        "modelfile": "Modelfile.fastAPI_Model",
        "setup_script": "setup_fastAPI_Model.ps1",
        "backend_env": {
            "LLM_PROVIDER": "ollama",
            "LLM_MODEL": CFG.model_name,
            "OLLAMA_NUM_CTX": "8192",
            "OLLAMA_AUTO_PULL": "false",
        },
    }
    (EXPORT_DIR / "MANIFEST.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )

    install_readme = f"""# Install the trained {CFG.model_name}

Produced on Colab. SHA-256 was computed on the exact bytes in this folder, so
`setup_fastAPI_Model.ps1` will reject a corrupted download rather than register
a damaged model.

## 1. Download from Drive

    training_runs/{RUN_SLUG}/export/
      {artifact_name}          ({artifact_size / 1e9:.2f} GB)
      Modelfile.fastAPI_Model
      MANIFEST.json

## 2. Put the files in place

Back up the current artifact first, then replace it:

```powershell
cd D:\\Projects\\git\\bot-training
Move-Item models\\{CFG.model_name}\\{artifact_name} `
          models\\{CFG.model_name}\\{artifact_name}.base-backup
Copy-Item <downloads>\\{artifact_name}  models\\{CFG.model_name}\\
Copy-Item <downloads>\\MANIFEST.json    models\\{CFG.model_name}\\
Copy-Item <downloads>\\Modelfile.fastAPI_Model  .
```

## 3. Register with Ollama

The script verifies size and SHA-256 against MANIFEST.json before registering:

```powershell
.\\setup_fastAPI_Model.ps1
```

## 4. Point the backend at it

`back-end-final\\.env`:

```env
LLM_PROVIDER=ollama
LLM_MODEL={CFG.model_name}
OLLAMA_HOST=http://localhost:11434
OLLAMA_NUM_CTX=8192
OLLAMA_AUTO_PULL=false
OLLAMA_AUTO_WARMUP=true
```

## 5. Verify against the real pipeline

```powershell
cd D:\\Projects\\git\\back-end-final
python scripts\\dev_server.py
python scripts\\live_project_scenarios.py
```

## Recorded results

- eval_loss: {final_eval.get("eval_loss")}
- overall contract rate: {report["overall"]["contract_rate"]}
- training minutes: {round(elapsed_minutes, 2)}
- GPU: {GPU_NAME}
- train rows: {len(train_records)} | eval rows: {len(eval_records)}

Per-stage numbers are in `../reports/eval_report.json`. Stages trained on one
seed record ({", ".join(t for t in ALL_TASKS if sum(1 for r in train_records if r["task"] == t) < 20)})
are not expected to improve; report them as untrained.
"""
    (EXPORT_DIR / "INSTALL.md").write_text(install_readme, encoding="utf-8")

    print()
    print("Export bundle ready on Drive:", EXPORT_DIR)
    print("  artifact:", artifact_name, f"({artifact_size / 1e9:.2f} GB)")
    print("  sha256:  ", artifact_hash)
    print("  files:   ", sorted(p.name for p in EXPORT_DIR.iterdir()))

    # The two 6.2 GB intermediates stay local by default. They are only worth
    # pushing through the Drive mount if you specifically need them off this
    # machine -- re-quantizing to another level, or pushing to the HF Hub.
    drive_free_gb = shutil.disk_usage(str(DRIVE_ROOT)).free / 1e9
    print()
    print(f"Drive free space: {drive_free_gb:.1f} GB")

    for enabled, label, source, target in (
        (CFG.copy_merged_to_drive, "merged fp16 weights", MERGED_DIR, EXPORT_DIR / "merged_hf"),
        (CFG.copy_f16_gguf_to_drive, "GGUF f16", GGUF_F16_PATH, EXPORT_DIR / GGUF_F16_PATH.name),
    ):
        if not enabled:
            print(f"  {label}: kept local only (set the matching CFG flag to copy)")
            continue
        if not source.exists():
            print(f"  {label}: not present, nothing to copy")
            continue
        size_gb = (
            sum(f.stat().st_size for f in source.rglob("*") if f.is_file())
            if source.is_dir()
            else source.stat().st_size
        ) / 1e9
        if drive_free_gb < size_gb * 1.1:
            print(f"  {label}: SKIPPED, needs {size_gb:.1f} GB but Drive has {drive_free_gb:.1f} GB")
            continue
        print(f"  {label}: copying {size_gb:.1f} GB to Drive, expect several minutes...")
        if target.exists():
            shutil.rmtree(target) if target.is_dir() else target.unlink()
        shutil.copytree(source, target) if source.is_dir() else shutil.copy2(source, target)
        drive_free_gb -= size_gb
        print(f"    done: {target}")
else:
    print("No quantized artifact to bundle.")
    if REHEARSAL:
        print("Expected: rehearsal skips merge and export. Set REHEARSAL = False for the artifact.")

Copying fastAPI_Model.Q4_K_M.gguf to Drive (about 2 GB, this takes a few minutes)...
Computing SHA-256 on the Drive copy...

Export bundle ready on Drive: /content/drive/MyDrive/bot-training/training_runs/fastAPI_Model-runtime-task/export
  artifact: fastAPI_Model.Q4_K_M.gguf (1.93 GB)
  sha256:   980C52AEC5DA1FA9A8DE56453B4C8CD85B9806BE850A1C22AB83E9C4B9640EF3
  files:    ['INSTALL.md', 'MANIFEST.json', 'Modelfile.fastAPI_Model', 'fastAPI_Model.Q4_K_M.gguf']

Drive free space: 7.5 GB
  merged fp16 weights: kept local only (set the matching CFG flag to copy)
  GGUF f16: kept local only (set the matching CFG flag to copy)


## 14. Run summary

In [8]:
run_summary = {
    "platform": "google-colab",
    "protocol": PROTOCOL["version"],
    "runtime_contract": PROTOCOL["runtime_contract"],
    "base_model": CFG.base_model_id,
    "gpu": GPU_NAME,
    "vram_mb": GPU_VRAM_MB,
    "compute_capability": GPU_COMPUTE,
    "memory_profile": CFG.memory_profile,
    "compute_dtype": str(COMPUTE_DTYPE),
    "rehearsal": REHEARSAL,
    "train_rows": len(train_records),
    "eval_rows": len(eval_records),
    "per_task_train": {task: sum(1 for r in train_records if r["task"] == task) for task in ALL_TASKS},
    "epochs": CFG.num_epochs,
    "max_seq_length": CFG.max_seq_length,
    "lora": {"r": CFG.lora_r, "alpha": CFG.lora_alpha, "dropout": CFG.lora_dropout},
    "train_minutes": round(elapsed_minutes, 2),
    "eval_loss": final_eval.get("eval_loss"),
    "eval_overall": report["overall"],
    "eval_per_task": report["per_task"],
    "export": export_state,
    "outputs": {
        "run_root": str(RUN_ROOT),
        "adapter": str(ADAPTER_DIR),
        "reports": str(REPORT_DIR),
        "export": str(EXPORT_DIR),
        "train_jsonl": str(TRAIN_JSONL),
        "eval_jsonl": str(EVAL_JSONL),
    },
}
(RUN_ROOT / "run_summary.json").write_text(
    json.dumps(run_summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(run_summary, indent=2, ensure_ascii=False))
print()
print("Everything above is on Drive under:", RUN_ROOT)

{
  "platform": "google-colab",
  "protocol": "runtime-task.v1",
  "runtime_contract": "project-run.v1",
  "base_model": "Qwen/Qwen2.5-Coder-3B-Instruct",
  "gpu": "cpu",
  "vram_mb": 0,
  "compute_capability": "n/a",
  "memory_profile": "GPU_16GB",
  "compute_dtype": "torch.float32",
  "rehearsal": false,
  "train_rows": 4247,
  "eval_rows": 642,
  "per_task_train": {
    "schema_plan": 1592,
    "api_contract": 1592,
    "file_plan": 1061,
    "create_file": 1,
    "edit_patch": 1,
    "validation_repair": 0
  },
  "epochs": 3.0,
  "max_seq_length": 1024,
  "lora": {
    "r": 32,
    "alpha": 64,
    "dropout": 0.05
  },
  "train_minutes": 240.34,
  "eval_loss": 0.0001055971224559471,
  "eval_overall": {
    "examples": 90,
    "parse_rate": 1.0,
    "contract_rate": 1.0,
    "exact_match_rate": 0.9778
  },
  "eval_per_task": {
    "file_plan": {
      "examples": 30,
      "parse_rate": 1.0,
      "contract_rate": 1.0,
      "exact_match_rate": 1.0,
      "component_f1": {
        "

### 14.1 Artifact inventory


In [9]:
print("=" * 78)
print("ARTIFACT INVENTORY")
print("=" * 78)


def describe(path: Path) -> str:
    if not path.exists():
        return "MISSING"
    size = (
        sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
        if path.is_dir()
        else path.stat().st_size
    )
    return f"{size / 1e9:.2f} GB"


rows = [
    ("LoRA adapter", ADAPTER_DIR, "Drive", "the trained weights themselves, ~1.9% of the model"),
    ("merged fp16 weights", MERGED_DIR, "local", "Hugging Face format; re-export or push to the Hub"),
    ("GGUF f16", GGUF_F16_PATH, "local", "re-quantize to any level without re-merging"),
    (f"GGUF {CFG.quantization}", GGUF_QUANT_PATH, "local", "the file the project actually runs"),
    (f"GGUF {CFG.quantization} on Drive", EXPORT_DIR / f"{CFG.model_name}.{CFG.quantization}.gguf",
     "Drive", "download this one"),
]
for label, path, where, why in rows:
    print(f"\n{label}")
    print(f"  size:  {describe(path)}   ({where})")
    print(f"  path:  {path}")
    print(f"  why:   {why}")

print()
print("=" * 78)
print("IMPORTANT: only the Drive copies survive the session.")
print("Anything marked 'local' disappears when the runtime is recycled.")
print("=" * 78)
missing_local = [
    label for label, path, where, _ in rows if where == "local" and not path.exists()
]
if missing_local:
    print("Not produced this run:", missing_local)
    print("Expected if REHEARSAL was True or the export cells were skipped.")
else:
    print("All three weight artifacts were produced.")
    print()
    print("To pull the two large ones off the machine, set these in section 3")
    print("and re-run section 13 (no retraining needed):")
    print("    CFG.copy_merged_to_drive = True")
    print("    CFG.copy_f16_gguf_to_drive = True")

ARTIFACT INVENTORY

LoRA adapter
  size:  0.25 GB   (Drive)
  path:  /content/drive/MyDrive/bot-training/training_runs/fastAPI_Model-runtime-task/lora_adapter
  why:   the trained weights themselves, ~1.9% of the model

merged fp16 weights
  size:  6.18 GB   (local)
  path:  /content/merged_hf
  why:   Hugging Face format; re-export or push to the Hub

GGUF f16
  size:  6.18 GB   (local)
  path:  /content/fastAPI_Model.f16.gguf
  why:   re-quantize to any level without re-merging

GGUF Q4_K_M
  size:  1.93 GB   (local)
  path:  /content/fastAPI_Model.Q4_K_M.gguf
  why:   the file the project actually runs

GGUF Q4_K_M on Drive
  size:  1.93 GB   (Drive)
  path:  /content/drive/MyDrive/bot-training/training_runs/fastAPI_Model-runtime-task/export/fastAPI_Model.Q4_K_M.gguf
  why:   download this one

IMPORTANT: only the Drive copies survive the session.
Anything marked 'local' disappears when the runtime is recycled.
All three weight artifacts were produced.

To pull the two large ones of